# Import libraries

In [ ]:
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import ListedColormap
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import BoundaryNorm
from matplotlib.gridspec import GridSpec
import cartopy.feature as cfeature
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cartopy.crs as ccrs
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib
import warnings
import glob
import os

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

## Combined Difference Mask

In [ ]:
# ======================================================
# COMMON ANALYSIS PERIOD
# ======================================================
time_slice = slice("1980-08-01", "2021-07-31")

# ======================================================
# DIRECTORIES
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_MODEL_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

RAIN_OBS_DIR   = "/glade/derecho/scratch/flehner/era5_daily/rain/"

OBS_ICE_FILES = ["/glade/work/xliu1/nasa_regrid/nasa_daily_1979_2000.nc",
                 "/glade/work/xliu1/nasa_regrid/nasa_daily_2001_2024.nc"]

OBS_SNOW_FILE = "/glade/work/xliu1/era5_snow_regrid/SM_snod_ERA5_01Aug1980-31Jul2021_v01_1deg.nc"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025"]

# ======================================================
# VARIABLES & THRESHOLDS
# ======================================================
ice_var  = "aice_d"
snow_var = "vsnon_d"
rain_var = "rain_d"

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh_obs   = 0.001   # mm/day
rain_thresh_model = 0.10

# ======================================================
# TIME CLEANER
# ======================================================
def fix_time_axis(da):
    da = da.sortby("time")
    _, index = np.unique(da["time"].values, return_index = True)
    return da.isel(time = index)

def fix_lon(da):
    da = da.assign_coords(lon = (((da.lon + 180) % 360) - 180))
    return da.sortby("lon")

# ======================================================
# LOAD OBSERVATIONS
# ======================================================

# --- ICE ---
ice_obs = xr.open_mfdataset(OBS_ICE_FILES, combine = "nested", concat_dim = "time", chunks = {"time":30})["cdr_seaice_conc"]

# --- SNOW ---
snow_obs = xr.open_dataset(OBS_SNOW_FILE, chunks = {"time":30})["snod"]

# --- NEW ERA5 RAIN ---
rain_files = sorted(glob.glob(f"{RAIN_OBS_DIR}/era5_daily_rain_*_r360x180.nc"))

# keep only 1980–2021
rain_files = [f for f in rain_files if any(str(y) in f for y in range(1980, 2022))]

def preprocess(ds):
    da = ds["rain"]
    da = da.sortby("time")
    _, idx = np.unique(da["time"].values, return_index = True)
    return da.isel(time = idx)

rain_obs = xr.open_mfdataset(rain_files, combine = "nested", concat_dim = "time", preprocess = preprocess, chunks = {"time": 30})

rain_obs = rain_obs.sortby("time")
_, idx = np.unique(rain_obs["time"].values, return_index = True)
rain_obs = rain_obs.isel(time = idx)

# ======================================================
# CLEAN + SUBSET
# ======================================================
ice_obs  = fix_time_axis(ice_obs)
snow_obs = fix_time_axis(snow_obs)

ice_obs  = fix_lon(ice_obs)
snow_obs = fix_lon(snow_obs)
rain_obs = fix_lon(rain_obs)

ice_obs  = ice_obs.sel(time = time_slice)
snow_obs = snow_obs.sel(time = time_slice)
rain_obs = rain_obs.sel(time = time_slice)

# ======================================================
# REGRID TO COMMON GRID
# ======================================================
ice_obs  = ice_obs.interp(lat = snow_obs.lat, lon = snow_obs.lon)
rain_obs = rain_obs.interp(lat = snow_obs.lat, lon = snow_obs.lon, kwargs = {"fill_value": None})

# Align
rain_obs = rain_obs.reindex(time = ice_obs.time, method = "nearest")
snow_obs = snow_obs.reindex(time = ice_obs.time, method = "nearest")
ice_obs, snow_obs, rain_obs = xr.align(ice_obs, snow_obs, rain_obs)

# ======================================================
# ROSI OBS
# ======================================================
rosi_obs = ((ice_obs  >= ice_thresh) & (snow_obs >= snow_thresh) & (rain_obs >= rain_thresh_obs))
rosi_obs = rosi_obs.fillna(False)
rosi_obs_freq = rosi_obs.mean("time")

# ======================================================
# MODEL
# ======================================================
def load_ros_part(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time":30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time":30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time":30}) as ds_r:

        ice  = fix_time_axis(ds_i[ice_var])
        snow = fix_time_axis(ds_s[snow_var])
        rain = fix_time_axis(ds_r[rain_var])

        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ice  = ice.sel(time = time_slice)
        snow = snow.sel(time = time_slice)
        rain = rain.sel(time = time_slice)

        if ice.time.size == 0:
            return None

        ros = ((ice  >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh_model))

        return ros

model_maps = []

for ens in ensemble_members:
    print("Processing ensemble:", ens)

    ros_chunks = []

    for p in periods:
        ros = load_ros_part(f"{ICE_DIR}/aice_{p}_{ens}.nc", f"{SNOW_DIR}/snow_{p}_{ens}.nc", f"{RAIN_MODEL_DIR}/rain_{p}_{ens}.nc")

        if ros is not None:
            ros_chunks.append(ros)

    if len(ros_chunks) == 0:
        continue

    ros_full = xr.concat(ros_chunks, dim = "time")
    ros_freq = ros_full.mean("time")

    model_maps.append(ros_freq)

model_maps = xr.concat(model_maps, dim = "ens")
model_mean = model_maps.mean("ens")

# ======================================================
# ALIGN & BIAS
# ======================================================
model_mean_interp = model_mean.interp(lat = rosi_obs_freq.lat, lon = rosi_obs_freq.lon)

bias = model_mean_interp - rosi_obs_freq

# ======================================================
# MAP SETTINGS
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

def setup_map(ax):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "lightgrey", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)
    return ax

# ======================================================
# DISCRETE COLOR LEVELS
# ======================================================
levels = [0, 0.001, 0.003, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]

cmap = plt.get_cmap("RdYlBu_r")
norm = BoundaryNorm(levels, cmap.N)

# ======================================================
# BIAS COLOR SETTINGS
# ======================================================
lim = 0.01

bias_levels = np.array([-lim, -0.008, -0.005, -0.003, 0, 0.003, 0.005, 0.008, lim])

n_bins = len(bias_levels) - 1
cmap_bias = plt.get_cmap("RdBu_r", n_bins)
norm_bias = BoundaryNorm(bias_levels, cmap_bias.N)

# ======================================================
# FIGURE LAYOUT
# ======================================================
fig = plt.figure(figsize = (18, 6))

gs = GridSpec(2, 4, figure = fig, height_ratios = [1, 0.05], width_ratios = [1, 1, 1, 0.05], wspace = 0.05)

ax1 = fig.add_subplot(gs[0, 0], projection = proj)
ax2 = fig.add_subplot(gs[0, 1], projection = proj)
ax3 = fig.add_subplot(gs[0, 2], projection = proj)

cax_bias = fig.add_subplot(gs[0, 3])
cax_bottom = fig.add_subplot(gs[1, 0:2])

setup_map(ax1)
setup_map(ax2)
setup_map(ax3)

# ------------------------------------------------------
# (a) Observations
# ------------------------------------------------------
p1 = ax1.pcolormesh(rosi_obs_freq.lon, rosi_obs_freq.lat, rosi_obs_freq, 
                    transform = transform, cmap = cmap, norm = norm, shading = "auto")

ax1.set_title("Observed Rain-on-Snow-on-Ice\n(ROSI) Frequency (1980–2021)", fontsize = 13)

ax1.text(0.02, 0.95, "(a)", transform = ax1.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# (b) Model
# ------------------------------------------------------
p2 = ax2.pcolormesh(model_mean_interp.lon, model_mean_interp.lat, model_mean_interp,
                    transform = transform, cmap = cmap, norm = norm, shading = "auto")

ax2.set_title("CESM2-LE Ensemble Mean\nROSI Frequency (1980–2021)", fontsize = 13)

ax2.text(0.02, 0.95, "(b)", transform = ax2.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# (c) Bias
# ------------------------------------------------------
p3 = ax3.pcolormesh(bias.lon, bias.lat, bias, transform = transform, cmap = cmap_bias, norm = norm_bias, shading = "auto")

ax3.set_title("CESM2-LE Bias (Model − Observations)", fontsize = 13)

ax3.text(0.02, 0.95, "(c)", transform = ax3.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# Shared colorbar for OBS + MODEL
# ------------------------------------------------------
cbar = fig.colorbar(p2, cax = cax_bottom, orientation = "horizontal", extend = "max", boundaries = levels)

cbar.set_label("ROSI Frequency (fraction of days annually)", fontsize = 12)
cbar.set_ticks(levels)

# ------------------------------------------------------
# Vertical colorbar for BIAS
# ------------------------------------------------------
cbar_bias = fig.colorbar(p3, cax = cax_bias, orientation = "vertical", extend = "both", boundaries = bias_levels)

cbar_bias.set_label("ROSI Frequency Bias (model − observations)", fontsize = 12)
cbar_bias.set_ticks(bias_levels)
cbar_bias.ax.tick_params(labelsize = 10)

# ------------------------------------------------------
# Save
# ------------------------------------------------------
plt.savefig("ROSI_obs_model_bias_1980_2021_rainfall.pdf", bbox_inches = 'tight')
# plt.savefig("ROSI_obs_model_bias_1980_2021_rainfall.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
# ======================================================
# EVENT DEFINITIONS
# ======================================================
soi_obs = ((ice_obs >= ice_thresh) & (snow_obs >= snow_thresh))

roi_obs = ((ice_obs >= ice_thresh) & (rain_obs >= rain_thresh_obs))

soi_obs = soi_obs.fillna(False)
roi_obs = roi_obs.fillna(False)

soi_obs_freq = soi_obs.mean("time")
roi_obs_freq = roi_obs.mean("time")

# ======================================================
# MODEL LOADER
# ======================================================
def load_components(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time":30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time":30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time":30}) as ds_r:

        ice  = fix_time_axis(ds_i["aice_d"])
        snow = fix_time_axis(ds_s["vsnon_d"])
        rain = fix_time_axis(ds_r["rain_d"])

        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ice  = ice.sel(time = time_slice)
        snow = snow.sel(time = time_slice)
        rain = rain.sel(time = time_slice)

        if ice.time.size == 0:
            return None, None

        soi = ((ice >= ice_thresh) & (snow >= snow_thresh))
        roi = ((ice >= ice_thresh) & (rain >= rain_thresh_model))

        return soi, roi

# ======================================================
# LOOP THROUGH ENSEMBLES
# ======================================================
soi_maps = []
roi_maps = []

for ens in ensemble_members:
    print("Processing ensemble:", ens)

    soi_chunks = []
    roi_chunks = []

    for p in periods:
        soi, roi = load_components(f"{ICE_DIR}/aice_{p}_{ens}.nc", f"{SNOW_DIR}/snow_{p}_{ens}.nc",
                                   f"{RAIN_MODEL_DIR}/rain_{p}_{ens}.nc")

        if soi is not None:
            soi_chunks.append(soi)
            roi_chunks.append(roi)

    if len(soi_chunks) == 0:
        continue

    soi_full = xr.concat(soi_chunks, dim = "time")
    roi_full = xr.concat(roi_chunks, dim = "time")

    soi_maps.append(soi_full.mean("time"))
    roi_maps.append(roi_full.mean("time"))

# Ensemble mean
soi_model = xr.concat(soi_maps, dim = "ens").mean("ens")
roi_model = xr.concat(roi_maps, dim = "ens").mean("ens")

# ======================================================
# INTERPOLATE TO OBS GRID
# ======================================================
soi_model_interp = soi_model.interp(lat = soi_obs_freq.lat, lon = soi_obs_freq.lon)
roi_model_interp = roi_model.interp(lat = roi_obs_freq.lat, lon = roi_obs_freq.lon)

# ======================================================
# BIAS (
# ======================================================
soi_bias = soi_model_interp - soi_obs_freq
roi_bias = roi_model_interp - roi_obs_freq

# ======================================================
# MASK (avoid zero-frequency dominance)
# ======================================================
soi_mask = soi_obs_freq > 1e-6
roi_mask = roi_obs_freq > 1e-6

# ======================================================
# METRICS (ROBUST + CONSISTENT)
# ======================================================

# Mean bias
soi_mean_bias = float(soi_bias.mean(["lat", "lon"], skipna = True))
roi_mean_bias = float(roi_bias.mean(["lat", "lon"], skipna = True))

# Mean absolute bias (MASKED — important)
soi_abs_bias = float(np.abs(soi_bias.where(soi_mask)).mean(["lat", "lon"], skipna = True))
roi_abs_bias = float(np.abs(roi_bias.where(roi_mask)).mean(["lat", "lon"], skipna = True))

# Weighted bias (focus on where events actually occur)
soi_weighted = float((soi_bias * soi_obs_freq).sum(["lat","lon"]) / (soi_obs_freq.sum(["lat","lon"]) + 1e-12))

roi_weighted = float((roi_bias * roi_obs_freq).sum(["lat","lon"]) / (roi_obs_freq.sum(["lat","lon"]) + 1e-12))

# Contribution fraction
total_abs = soi_abs_bias + roi_abs_bias + 1e-12
soi_frac = soi_abs_bias / total_abs
roi_frac = roi_abs_bias / total_abs

# ======================================================
# TABLE
# ======================================================
df = pd.DataFrame({"Metric": ["Mean Bias", "Mean Absolute Bias", "Weighted Bias", "Contribution Fraction"],
                   "SOI": [soi_mean_bias, soi_abs_bias, soi_weighted, soi_frac],
                   "ROI": [roi_mean_bias, roi_abs_bias, roi_weighted, roi_frac]})

df_fmt = df.copy()
df_fmt["SOI"] = df_fmt["SOI"].map(lambda x: f"{x:.4f}")
df_fmt["ROI"] = df_fmt["ROI"].map(lambda x: f"{x:.4f}")

print("\n=== Contribution Summary ===")
print(df_fmt.to_string(index = False))

# Save
df.to_csv("SOI_ROI_contribution_summary.csv", index = False)

In [ ]:
print("SOI obs mean:", float(soi_obs_freq.mean(["lat","lon"])))
print("ROI obs mean:", float(roi_obs_freq.mean(["lat","lon"])))

print("SOI bias mean:", float(soi_bias.mean(["lat","lon"])))
print("ROI bias mean:", float(roi_bias.mean(["lat","lon"])))

## CESM2 Change + SNR

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# DIRECTORIES
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

# ======================================================
# THRESHOLDS
# ======================================================
ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10
snr_threshold = 2.0

# ======================================================
# MAP SETTINGS
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

# ======================================================
# LOAD FUNCTION — RETURNS ANNUAL FRACTION
# ======================================================
def annual_ros_fraction(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time": 30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time": 30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time": 30}) as ds_r:

        ice  = ds_i["aice_d"]
        snow = ds_s["vsnon_d"]
        rain = ds_r["rain_d"]

        # remove duplicate CESM times
        ice  = ice.isel(time = ~ice.get_index("time").duplicated())
        snow = snow.isel(time = ~snow.get_index("time").duplicated())
        rain = rain.isel(time = ~rain.get_index("time").duplicated())

        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ros = (ice >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh)

        # ---- CRITICAL: KEEP YEAR DIMENSION ----
        yearly = ros.groupby("time.year").mean("time")

        return yearly

# ======================================================
# LOAD ALL ENSEMBLES
# ======================================================
ros_all = {p:[] for p in periods}

for ens in ensemble_members:
    print("Processing ensemble:",ens)
    for p in periods:
        ros_all[p].append(annual_ros_fraction(
            f"{ICE_DIR}/aice_{p}_{ens}.nc", f"{SNOW_DIR}/snow_{p}_{ens}.nc", f"{RAIN_DIR}/rain_{p}_{ens}.nc"))

for p in periods:
    ros_all[p] = xr.concat(ros_all[p], dim = "ens").assign_coords(ens = ensemble_members)

lon = ros_all[periods[0]].lon
lat = ros_all[periods[0]].lat

# ======================================================
# SIGNAL / NOISE
# ======================================================
mean1_ens = ros_all["1970-2000"].mean("year")
mean2_ens = ros_all["2000-2025"].mean("year")
mean4_ens = ros_all["2065-2100"].mean("year")

signal1 = mean2_ens.mean("ens") - mean1_ens.mean("ens")
signal3 = mean4_ens.mean("ens") - mean1_ens.mean("ens")

noise1 = (mean2_ens - mean1_ens).std("ens", ddof = 1)
noise3 = (mean4_ens - mean1_ens).std("ens", ddof = 1)

eps = 1e-6
snr1 = signal1 / noise1.where(noise1 > eps)
snr3 = signal3 / noise3.where(noise3 > eps)

mask1 = np.abs(snr1) > snr_threshold
mask3 = np.abs(snr3) > snr_threshold

panel_labels = ["(a)", "(b)"]

# ======================================================
# STIPPLE FUNCTION 
# ======================================================
def add_stipple(ax, lon_da, lat_da, mask_da, jitter = 0.2, size = 2, alpha = 0.5, color = "tomato", seed = 42):

    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)
    if len(xs) == 0:
        return

    if getattr(lon_da, "ndim", None) == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    sig_lon = sig_lon + np.random.uniform(-jitter, jitter, len(sig_lon))
    sig_lat = sig_lat + np.random.uniform(-jitter, jitter, len(sig_lat))

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 30)

# ======================================================
# FIGURE 1 — CHANGE MAPS + STIPPLING
# ======================================================
maps = [signal1, signal3]
masks = [mask1, mask3]
titles = ["Rain-on-Snow-on-Ice (ROSI) Mean\nFrequency Change(2000–2025 − 1970–2000)",
          "ROSI Mean Frequency Change\n(2065–2100 - 1970–2000)"]

vmin = min([m.min() for m in maps]).compute().item()
vmax = max([m.max() for m in maps]).compute().item()

lim = max(abs(vmin), abs(vmax))
lim = np.ceil(lim * 100) / 100   # round to 0.01

# -----------------------------------------
# symmetric bins with TRUE zero boundary
# -----------------------------------------
n_side = 5

neg = np.linspace(-lim, 0, n_side + 1)
pos = np.linspace(0, lim, n_side + 1)[1:]   # remove duplicate zero

levels = np.concatenate([neg, pos])
levels = np.round(levels, 3)

n_bins = len(levels) - 1

cmap = plt.get_cmap("RdBu_r", n_bins)
norm = BoundaryNorm(levels, cmap.N)

fig1, axs1 = plt.subplots(1, 2, figsize = (8, 4), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax in axs1:
    ax.set_facecolor("#f7f7f7")

for ax, data, mask, title, label in zip(axs1, maps, masks, titles, panel_labels):

    ax.set_extent(extent,crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "lightgrey", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p1 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = cmap, norm = norm, shading = "auto")

    add_stipple(ax, lon, lat, mask)
    ax.set_title(title)
    ax.text(0.02, 0.95, label, transform = ax.transAxes, fontsize = 14, fontweight = "bold",va = "top", ha = "left", 
            bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

ticks = levels

cbar1 = fig1.colorbar(p1, ax = axs1, orientation = "horizontal", pad = 0.04, shrink = 0.7, 
                      fraction = 0.05, aspect = 40, extend = "both", boundaries = levels, ticks = ticks)

cbar1.set_label("ROSI Frequency Change (fraction of days annually)", fontsize = 12)
cbar1.ax.tick_params(labelsize = 10)
cbar1.ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))

fig1.savefig("cesm2_rosi_mean_change.pdf", bbox_inches = 'tight')
# fig1.savefig("cesm2_rosi_mean_change.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ======================================================
# FIGURE 2 — SNR MAPS
# ======================================================
snr_maps = [snr1, snr3]

vmin = min([m.min() for m in snr_maps]).compute().item()
vmax = max([m.max() for m in snr_maps]).compute().item()

# --- clean integer-centered SNR bins ---
lim = 5

levels_snr = np.arange(-lim, lim + 1, 1)

cmap_snr = plt.get_cmap("RdBu_r", len(levels_snr) - 1)
norm_snr = BoundaryNorm(levels_snr, cmap_snr.N)

titles_snr = ["ROSI Frequency Change Signal-to-Noise Ratio\n(SNR) (2000–2025 − 1970–2000)", 
              "ROSI Frequency Change SNR\n(2065–2100 - 1970–2000)"]
ticks_snr = [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]

fig2, axs2 = plt.subplots(1, 2, figsize = (8, 4), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax, data, title, label in zip(axs2, snr_maps, titles_snr, panel_labels):

    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "lightgrey", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = cmap_snr, norm = norm_snr, shading = "auto")

    ax.contour(lon, lat, data, levels= [-2, 2], colors = "black", linewidths = 0.8, transform = transform)

    ax.text(0.02, 0.95, label, transform = ax.transAxes, fontsize = 14, fontweight = "bold", va = "top", ha = "left",
            bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))
    
    ax.set_title(title, fontsize = 13)

cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.04, shrink = 0.7, 
                      fraction = 0.05, aspect = 40, extend = "both", ticks = ticks_snr)

cbar2.set_label("ROSI Mean Change SNR", fontsize = 12)
cbar2.ax.tick_params(labelsize = 9)

fig2.savefig("cesm2_rosi_snr.pdf", bbox_inches = 'tight')
# fig2.savefig("cesm2_rosi_snr.png", dpi = 300, bbox_inches = "tight")
plt.show()

## Mean Individual Components

In [ ]:
# ======================================================
# LOAD FUNCTION — RETURNS ANNUAL MEAN
# ======================================================
def annual_mean_component(file_path, varname):

    with xr.open_dataset(file_path, chunks = {"time":30}) as ds:
        data = ds[varname]

        data = data.isel(time = ~data.get_index("time").duplicated())

        yearly = data.groupby("time.year").mean("time")

        return yearly

ice_all  = {p:[] for p in periods}
snow_all = {p:[] for p in periods}
rain_all = {p:[] for p in periods}

for ens in ensemble_members:
    print("Component processing ensemble:", ens)
    for p in periods:
        ice_all[p].append(annual_mean_component(f"{ICE_DIR}/aice_{p}_{ens}.nc", "aice_d"))
        snow_all[p].append(annual_mean_component(f"{SNOW_DIR}/snow_{p}_{ens}.nc", "vsnon_d"))
        rain_all[p].append(annual_mean_component(f"{RAIN_DIR}/rain_{p}_{ens}.nc", "rain_d"))

for p in periods:
    ice_all[p]  = xr.concat(ice_all[p],  dim = "ens").assign_coords(ens = ensemble_members)
    snow_all[p] = xr.concat(snow_all[p], dim = "ens").assign_coords(ens = ensemble_members)
    rain_all[p] = xr.concat(rain_all[p], dim = "ens").assign_coords(ens = ensemble_members)

# Mean over year first
ice_base  = ice_all["1970-2000"].mean("year")
ice_fut   = ice_all["2065-2100"].mean("year")

snow_base = snow_all["1970-2000"].mean("year")
snow_fut  = snow_all["2065-2100"].mean("year")

rain_base = rain_all["1970-2000"].mean("year")
rain_fut  = rain_all["2065-2100"].mean("year")

# Ensemble mean difference
d_ice  = ice_fut.mean("ens")  - ice_base.mean("ens")
d_snow = snow_fut.mean("ens") - snow_base.mean("ens")
d_rain = rain_fut.mean("ens") - rain_base.mean("ens")

maps_comp = [d_rain, d_snow, d_ice]
titles_comp = ["CESM2-LE Mean Change in Rainfall\n(2065–2100 − 1970–2000)",
               "CESM2-LE Mean Change in Snow Depth\n(2065–2100 − 1970–2000)",
               "CESM2-LE Mean Change in Sea Ice Fraction\n(2065–2100 − 1970–2000)"]

panel_labels = ["(a)", "(b)", "(c)"]

# ======================================================
# Symmetric DISCRETE levels with white-centered zero
# ======================================================
fig3, axs3 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj))

for ax, data, title, label in zip(axs3, maps_comp, titles_comp, panel_labels):

    # --- symmetric scaling PER PANEL ---
    vmin = float(data.min().compute())
    vmax = float(data.max().compute())
    
    lim = max(abs(vmin), abs(vmax))
    if "Rainfall" in title:
        lim = np.ceil(lim * 100) / 100

    elif "Snow" in title:
        lim = np.ceil(lim * 100) / 100
    
    else:   # ice concentration
        lim = np.ceil(lim * 10) / 10
    
    # --- symmetric bins with true zero boundary ---
    n_side = 5
    
    neg = np.linspace(-lim, 0, n_side + 1)
    pos = np.linspace(0, lim, n_side + 1)[1:]
    
    levels = np.concatenate([neg, pos])
    levels = np.round(levels, 3)
    
    cmap = plt.get_cmap("RdBu_r", len(levels) - 1)
    norm = BoundaryNorm(levels, cmap.N)

    # --- map ---
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "lightgrey", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p3 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = cmap, norm = norm, shading = "auto")

    ax.text(0.02, 0.95, label, transform = ax.transAxes, fontsize = 15, fontweight = "bold", va = "top", ha = "left",
            bbox =  dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

    ax.set_title(title, fontsize = 16)

    # --- individual colorbar ---
    cbar = fig3.colorbar(p3, ax = ax, orientation = "horizontal", pad = 0.05, shrink = 0.8, aspect = 30, ticks = levels)

    if "Rainfall" in title:
        cbar.set_label("Mean Rainfall Change (cm day⁻¹)", fontsize = 12)
    elif "Snow Depth" in title:
        cbar.set_label("Mean Snow Depth Change (m)", fontsize = 12)
    else:
        cbar.set_label("Mean Sea Ice Fraction Change", fontsize = 12)
    cbar.ax.tick_params(labelsize = 9)

fig3.tight_layout()
fig3.savefig("cesm2_component_change_mechanism.pdf", bbox_inches = 'tight')
# fig3.savefig("cesm2_component_change_mechanism.png", dpi = 300, bbox_inches = "tight")

plt.show()

## CESM2 Time Series

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# TIME SERIES (ALL 10)
# ======================================================

ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10

# ======================================================
# FUNCTION: Compute annual RoS frequency for one file set
# ======================================================
def load_member_data(ice_file, snow_file, rain_file):

    ds_i = xr.open_dataset(ice_file, chunks = {"time": 365})
    ds_s = xr.open_dataset(snow_file, chunks = {"time": 365})
    ds_r = xr.open_dataset(rain_file, chunks = {"time": 365})

    ice  = ds_i["aice_d"]
    snow = ds_s["vsnon_d"]
    rain = ds_r["rain_d"]

    ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

    return ice, snow, rain
    
# ======================================================
# BUILD ENSEMBLE TIMESERIES
# ======================================================
ros_ensemble = []

for ens in ensemble_members:

    print(f"Processing ensemble {ens}")

    ice_all  = []
    snow_all = []
    rain_all = []

    for period in periods:

        ice_file  = os.path.join(ICE_DIR,  f"aice_{period}_{ens}.nc")
        snow_file = os.path.join(SNOW_DIR, f"snow_{period}_{ens}.nc")
        rain_file = os.path.join(RAIN_DIR, f"rain_{period}_{ens}.nc")

        ice, snow, rain = load_member_data(ice_file, snow_file, rain_file)

        ice_all.append(ice)
        snow_all.append(snow)
        rain_all.append(rain)

    # 🔥 concatenate DAILY first
    ice  = xr.concat(ice_all, dim = "time").sortby("time")
    snow = xr.concat(snow_all, dim = "time").sortby("time")
    rain = xr.concat(rain_all, dim = "time").sortby("time")

    # remove duplicate timestamps
    ice  = ice.sel(time = ~ice.get_index("time").duplicated())
    snow = snow.sel(time = ~snow.get_index("time").duplicated())
    rain = rain.sel(time = ~rain.get_index("time").duplicated())

    # --- static mask ---
    valid_mask = (~np.isnan(ice.isel(time = 0))) & (~np.isnan(snow.isel(time = 0))) & (~np.isnan(rain.isel(time = 0)))

    # --- ROS daily ---
    ros_daily = ((ice >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh))

    ros_daily = ros_daily & valid_mask

    # 🔥 resample ONLY ONCE
    ros_yearly = ros_daily.resample(time = "YE").any()

    # --- weights ---
    weights = np.cos(np.deg2rad(ice.lat))
    weights = xr.DataArray(weights, dims = ["lat"])
    weights, _ = xr.broadcast(weights, ros_yearly)
    weights = weights.where(valid_mask)

    fraction = (ros_yearly * weights).sum(dim = ("lat", "lon")) / weights.sum(dim = ("lat", "lon"))

    ros_ensemble.append(fraction)
    
# Stack ensembles
ros_ts_all = xr.concat(ros_ensemble, dim = "ens", join = "inner")
    
if not isinstance(ros_ts_all.indexes["time"], pd.DatetimeIndex):
    ros_ts_all = ros_ts_all.assign_coords(time = ros_ts_all.indexes["time"].to_datetimeindex())
    
ros_ts_all = ros_ts_all.sel(time = slice("1970", "2100"))
    
mean_ts = ros_ts_all.mean("ens")
std_ts = ros_ts_all.std("ens")

# ======================================================
# PLOT
# ======================================================
plt.figure(figsize = (11, 5))

time = mean_ts["time"].values

# ------------------------------------------------------
# Individual ensemble members
# ------------------------------------------------------
for i in range(ros_ts_all.sizes["ens"]):
    plt.plot(time, ros_ts_all.isel(ens = i).values, alpha = 0.15, linewidth = 1)

# ------------------------------------------------------
# Ensemble mean
# ------------------------------------------------------
plt.plot(time, mean_ts.values, linewidth = 2.5, label = "Ensemble Mean")

# ------------------------------------------------------
# Ensemble spread (±1σ)
# ------------------------------------------------------
plt.fill_between(time, (mean_ts - std_ts).values, (mean_ts + std_ts).values, alpha = 0.3, label = "Ensemble Spread (±1σ)")

plt.ylabel("Annual Fraction of Arctic Area Experiencing\n≥1 Rain-on-Snow-on-Ice (ROSI) Event", size = 12)
plt.xlabel("Year")
plt.grid(alpha = 0.2)
plt.legend()
plt.tight_layout()
plt.savefig("cesm2_rosi_timeseries.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# TIME SERIES (ALL 10)
# ======================================================

ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10

# ======================================================
# FUNCTION: Compute annual RoS frequency for one file set
# ======================================================
def load_member_data(ice_file, snow_file, rain_file):

    ds_i = xr.open_dataset(ice_file, chunks = {"time": 365})
    ds_s = xr.open_dataset(snow_file, chunks = {"time": 365})
    ds_r = xr.open_dataset(rain_file, chunks = {"time": 365})

    ice  = ds_i["aice_d"]
    snow = ds_s["vsnon_d"]
    rain = ds_r["rain_d"]

    ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

    return ice, snow, rain

# ======================================================
# BUILD TEMPORAL ROSI TIMESERIES (FINAL)
# ======================================================
rosi_ensemble = []

for ens in ensemble_members:

    print(f"Processing ensemble {ens}")

    rosi_years = []

    for period in periods:

        ice_file  = os.path.join(ICE_DIR,  f"aice_{period}_{ens}.nc")
        snow_file = os.path.join(SNOW_DIR, f"snow_{period}_{ens}.nc")
        rain_file = os.path.join(RAIN_DIR, f"rain_{period}_{ens}.nc")

        ice, snow, rain = load_member_data(ice_file, snow_file, rain_file)

        # ----------------------------
        # MASKS
        # ----------------------------
        valid_mask = (~np.isnan(ice.isel(time = 0))) & (~np.isnan(snow.isel(time = 0))) & (~np.isnan(rain.isel(time = 0)))

        ice_mask  = (ice >= ice_thresh)
        snow_mask = (snow >= snow_thresh)
        rain_mask = (rain >= rain_thresh)

        # ----------------------------
        # ROSI CONDITION
        # ----------------------------
        rosi = ice_mask & snow_mask & rain_mask & valid_mask

        # ----------------------------
        # WEIGHTS
        # ----------------------------
        weights = np.cos(np.deg2rad(ice.lat))
        weights = xr.DataArray(weights, dims = ["lat"])
        weights, _ = xr.broadcast(weights, ice_mask)
        weights = weights.where(valid_mask)

        # ----------------------------
        # DAILY FRACTION
        # ----------------------------
        num = (rosi * weights).sum(dim = ("lat", "lon"))
        den = (ice_mask * weights).sum(dim = ("lat", "lon"))

        # stabilize denominator
        den_smooth = den.rolling(time = 30, center = True, min_periods = 10).mean()
        den_smooth = den_smooth.chunk({"time": -1})
        min_ice_area = den_smooth.quantile(0.05)

        frac = (num / den_smooth).where(den_smooth > min_ice_area)

        rosi_years.append(frac)

        # free memory
        del ice, snow, rain
        del ice_mask, snow_mask, rain_mask, rosi
        del num, den, frac

    # ----------------------------
    # CONCAT + RESAMPLE
    # ----------------------------
    rosi_daily = xr.concat(rosi_years, dim = "time").sortby("time")

    rosi_daily = rosi_daily.sel(time = ~rosi_daily.get_index("time").duplicated())

    rosi_member = rosi_daily.resample(time = "YE").mean().compute()

    rosi_ensemble.append(rosi_member)

# ======================================================
# STACK
# ======================================================
rosi_ts = xr.concat(rosi_ensemble, dim = "ens")

# 🔥 FIX: convert cftime → pandas datetime
if not isinstance(rosi_ts.indexes["time"], pd.DatetimeIndex):
    rosi_ts = rosi_ts.assign_coords(time = rosi_ts.indexes["time"].to_datetimeindex())

rosi_ts = rosi_ts.sel(time = slice("1970", "2100"))

rosi_mean = rosi_ts.mean("ens")
rosi_std  = rosi_ts.std("ens")

# optional smoothing
rosi_mean = rosi_mean.rolling(time = 3, center = True).mean()

# ======================================================
# PLOT
# ======================================================
plt.figure(figsize = (11, 5))

time = rosi_mean["time"].values

for i in range(rosi_ts.sizes["ens"]):
    plt.plot(time, rosi_ts.isel(ens = i).values, alpha = 0.15, linewidth = 1)

plt.plot(time, rosi_mean.values, linewidth = 2.5, label = "Ensemble Mean")
plt.fill_between(time, (rosi_mean - rosi_std).values, (rosi_mean + rosi_std).values, alpha = 0.3, label = "Ensemble Spread (±1σ)")

plt.ylabel("Mean Daily Fraction of Sea Ice Area Experiencing\nRain-on-Snow-on-Ice (ROSI) Event", size = 12)
plt.xlabel("Year")
plt.grid(alpha = 0.2)
plt.legend()

plt.tight_layout()
plt.savefig("cesm2_rosi_temporal_timeseries.png", dpi = 300)
plt.show()

## Individual Time Series

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# PATHS + SETTINGS
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10

# ======================================================
# FUNCTION: LOAD DATA
# ======================================================
def load_member_data(ice_file, snow_file, rain_file):

    ds_i = xr.open_dataset(ice_file, chunks = {"time": 365})
    ds_s = xr.open_dataset(snow_file, chunks = {"time": 365})
    ds_r = xr.open_dataset(rain_file, chunks = {"time": 365})

    ice  = ds_i["aice_d"]
    snow = ds_s["vsnon_d"]
    rain = ds_r["rain_d"]

    ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

    return ice, snow, rain

# ======================================================
# BUILD ENSEMBLE TIMESERIES
# ======================================================
snow_ice_ensemble = []
rain_ice_ensemble = []

for ens in ensemble_members:

    print(f"Processing ensemble {ens}")

    ice_all  = []
    snow_all = []
    rain_all = []

    for period in periods:

        ice_file  = os.path.join(ICE_DIR,  f"aice_{period}_{ens}.nc")
        snow_file = os.path.join(SNOW_DIR, f"snow_{period}_{ens}.nc")

        rain_file = os.path.join(RAIN_DIR, f"rain_{period}_{ens}.nc")

        ice, snow, rain = load_member_data(ice_file, snow_file, rain_file)

        ice_all.append(ice)
        snow_all.append(snow)
        rain_all.append(rain)

    # ----------------------------------------
    # CONCAT DAILY FIRST
    # ----------------------------------------
    ice  = xr.concat(ice_all, dim = "time").sortby("time")
    snow = xr.concat(snow_all, dim = "time").sortby("time")
    rain = xr.concat(rain_all, dim = "time").sortby("time")

    # remove duplicate timestamps
    ice  = ice.sel(time = ~ice.get_index("time").duplicated())
    snow = snow.sel(time = ~snow.get_index("time").duplicated())
    rain = rain.sel(time = ~rain.get_index("time").duplicated())

    # ----------------------------------------
    # STATIC VALID MASK
    # ----------------------------------------
    valid_mask = (~np.isnan(ice.isel(time = 0))) & (~np.isnan(snow.isel(time = 0))) & (~np.isnan(rain.isel(time = 0)))

    # ----------------------------------------
    # CONDITIONS
    # ----------------------------------------
    ice_mask  = (ice  >= ice_thresh)
    snow_mask = (snow >= snow_thresh)
    rain_mask = (rain >= rain_thresh)

    snow_on_ice = ice_mask & snow_mask & valid_mask
    rain_on_ice = ice_mask & rain_mask & valid_mask

    # ----------------------------------------
    # YEARLY "AT LEAST ONE EVENT"
    # ----------------------------------------
    ice_yearly        = ice_mask.resample(time = "YE").any()
    ice_yearly = ice_yearly & valid_mask
    snow_on_ice_yearly = snow_on_ice.resample(time = "YE").any()
    rain_on_ice_yearly = rain_on_ice.resample(time = "YE").any()

    # ----------------------------------------
    # AREA WEIGHTS
    # ----------------------------------------
    weights = np.cos(np.deg2rad(ice.lat))
    weights = xr.DataArray(weights, dims = ["lat"])
    weights, _ = xr.broadcast(weights, ice_yearly)

    weights = weights.where(valid_mask)

    # ----------------------------------------
    # FRACTION OVER ICE AREA
    # ----------------------------------------
    numerator = (snow_on_ice_yearly * weights).sum(dim = ("lat", "lon"))
    denominator = weights.sum(dim = ("lat", "lon"))

    fraction = numerator / denominator

    snow_ice_ensemble.append(fraction)

    numerator_rain = (rain_on_ice_yearly * weights).sum(dim = ("lat","lon"))
    
    fraction_rain_on_ice = numerator_rain / denominator

    rain_ice_ensemble.append(fraction_rain_on_ice)

# ======================================================
# STACK ENSEMBLES
# ======================================================
snow_ice_ts = xr.concat(snow_ice_ensemble, dim = "ens")
rain_ice_ts = xr.concat(rain_ice_ensemble, dim = "ens")

# ensure datetime
if not isinstance(snow_ice_ts.indexes["time"], pd.DatetimeIndex):
    snow_ice_ts = snow_ice_ts.assign_coords(time = snow_ice_ts.indexes["time"].to_datetimeindex(time_unit = "ns"))

if not isinstance(rain_ice_ts.indexes["time"], pd.DatetimeIndex):
    rain_ice_ts = rain_ice_ts.assign_coords(time = rain_ice_ts.indexes["time"].to_datetimeindex(time_unit = "ns"))

# slice BOTH
snow_ice_ts = snow_ice_ts.sel(time = slice("1970", "2100"))
rain_ice_ts = rain_ice_ts.sel(time = slice("1970", "2100"))

# 🔥 align them (best practice)
snow_ice_ts, rain_ice_ts = xr.align(snow_ice_ts, rain_ice_ts, join = "inner")

# ======================================================
# STATS
# ======================================================
snow_mean = snow_ice_ts.mean("ens")
snow_std  = snow_ice_ts.std("ens")

rain_mean = rain_ice_ts.mean("ens")
rain_std  = rain_ice_ts.std("ens")

# ======================================================
# PLOT (COMBINED)
# ======================================================
fig, ax = plt.subplots(figsize = (11, 5))

time = snow_mean["time"].values

# ----------------------------------------
# SNOW ON ICE (blue)
# ----------------------------------------
for i in range(snow_ice_ts.sizes["ens"]):
    ax.plot(time, snow_ice_ts.isel(ens = i), color = "blue", alpha = 0.08, linewidth = 1)

ax.plot(time, snow_mean, color = "blue", linewidth = 2.5, label = "Snow-on-ice (≥1 event/yr)")
ax.fill_between(time, snow_mean - snow_std, snow_mean + snow_std, color = "blue", alpha = 0.25)

# ----------------------------------------
# RAIN ON ICE (orange)
# ----------------------------------------
for i in range(rain_ice_ts.sizes["ens"]):
    ax.plot(time, rain_ice_ts.isel(ens = i), color = "tab:orange", alpha = 0.08, linewidth = 1)

ax.plot(time, rain_mean, color = "tab:orange", linewidth = 2.5, label = "Rain-on-ice (≥1 event/yr)")

ax.fill_between(time, rain_mean - rain_std, rain_mean + rain_std, color = "tab:orange", alpha = 0.25)

# ----------------------------------------
# LABELS + LEGEND
# ----------------------------------------
ax.set_ylabel("Annual Fraction of Arctic Area Experiencing\n≥1 Rain-on-Snow-on-Ice (ROSI) Event", size = 13)
ax.set_xlabel("Year")

ax.legend(frameon = False)
ax.grid(alpha = 0.2)

# ======================================================
# FINALIZE
# ======================================================
plt.tight_layout()
plt.savefig("cesm2_snow_rain_on_ice_combined.png", dpi = 300)
plt.show()

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================
# PATHS + SETTINGS
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10

# ======================================================
# FUNCTION: LOAD DATA
# ======================================================
def load_member_data(ice_file, snow_file, rain_file):

    ds_i = xr.open_dataset(ice_file, chunks={"time": 30})
    ds_s = xr.open_dataset(snow_file, chunks={"time": 30})
    ds_r = xr.open_dataset(rain_file, chunks={"time": 30})

    ice  = ds_i["aice_d"]
    snow = ds_s["vsnon_d"]
    rain = ds_r["rain_d"]

    ice, snow, rain = xr.align(ice, snow, rain, join="inner")

    return ice, snow, rain

# ======================================================
# BUILD ENSEMBLE TIMESERIES
# ======================================================
snow_ice_ensemble = []
rain_ice_ensemble = []

for ens in ensemble_members:

    print(f"Processing ensemble {ens}")

    ice_all  = []
    snow_all = []
    rain_all = []

    # ----------------------------------------
    # LOAD + CONCAT FIRST (CRITICAL FIX)
    # ----------------------------------------
    for period in periods:

        ice_file  = os.path.join(ICE_DIR,  f"aice_{period}_{ens}.nc")
        snow_file = os.path.join(SNOW_DIR, f"snow_{period}_{ens}.nc")
        rain_file = os.path.join(RAIN_DIR, f"rain_{period}_{ens}.nc")

        ice, snow, rain = load_member_data(ice_file, snow_file, rain_file)

        ice_all.append(ice)
        snow_all.append(snow)
        rain_all.append(rain)

    ice  = xr.concat(ice_all, dim="time").sortby("time")
    snow = xr.concat(snow_all, dim="time").sortby("time")
    rain = xr.concat(rain_all, dim="time").sortby("time")

    # remove duplicate timestamps
    ice  = ice.sel(time=~ice.get_index("time").duplicated())
    snow = snow.sel(time=~snow.get_index("time").duplicated())
    rain = rain.sel(time=~rain.get_index("time").duplicated())

    # ----------------------------------------
    # MASKS
    # ----------------------------------------
    valid_mask = (~np.isnan(ice)) & (~np.isnan(snow)) & (~np.isnan(rain))

    ice_mask  = (ice  >= ice_thresh)
    snow_mask = (snow >= snow_thresh)
    rain_mask = (rain >= rain_thresh)

    # ----------------------------------------
    # WEIGHTS
    # ----------------------------------------
    weights = np.cos(np.deg2rad(ice.lat))
    weights = xr.DataArray(weights, dims=["lat"])
    weights, _ = xr.broadcast(weights, ice)

    weights = weights.where(valid_mask)

    # ----------------------------------------
    # EVENTS
    # ----------------------------------------
    snow_on_ice = ice_mask & snow_mask & valid_mask
    rain_on_ice = ice_mask & rain_mask & valid_mask

    # ----------------------------------------
    # DAILY FRACTIONS
    # ----------------------------------------
    num_snow = (snow_on_ice * weights).sum(dim=("lat", "lon"))
    num_rain = (rain_on_ice * weights).sum(dim=("lat", "lon"))
    den      = (ice_mask * weights).sum(dim=("lat", "lon"))

    # ----------------------------------------
    # HANDLE LOW ICE AREA (CLEAN FIX)
    # ----------------------------------------
    threshold = den.max(dim="time") * 0.01   # 1% of max ice area

    frac_snow = (num_snow / den).where(den > threshold)
    frac_rain = (num_rain / den).where(den > threshold)

    # ----------------------------------------
    # YEARLY MEAN
    # ----------------------------------------
    snow_member = frac_snow.resample(time="YE").mean().compute()
    rain_member = frac_rain.resample(time="YE").mean().compute()

    snow_ice_ensemble.append(snow_member)
    rain_ice_ensemble.append(rain_member)

# ======================================================
# STACK ENSEMBLES
# ======================================================
snow_ice_ts = xr.concat(snow_ice_ensemble, dim="ens")
rain_ice_ts = xr.concat(rain_ice_ensemble, dim="ens")

# ensure datetime
if not isinstance(snow_ice_ts.indexes["time"], pd.DatetimeIndex):
    snow_ice_ts = snow_ice_ts.assign_coords(
        time=snow_ice_ts.indexes["time"].to_datetimeindex(time_unit="ns")
    )

if not isinstance(rain_ice_ts.indexes["time"], pd.DatetimeIndex):
    rain_ice_ts = rain_ice_ts.assign_coords(
        time=rain_ice_ts.indexes["time"].to_datetimeindex(time_unit="ns")
    )

# slice
snow_ice_ts = snow_ice_ts.sel(time=slice("1970", "2100"))
rain_ice_ts = rain_ice_ts.sel(time=slice("1970", "2100"))

# align
snow_ice_ts, rain_ice_ts = xr.align(snow_ice_ts, rain_ice_ts, join="inner")

# ======================================================
# STATS
# ======================================================
snow_mean = snow_ice_ts.mean("ens")
snow_std  = snow_ice_ts.std("ens")

rain_mean = rain_ice_ts.mean("ens")
rain_std  = rain_ice_ts.std("ens")

# optional smoothing (SAFE)
snow_mean = snow_mean.rolling(time=3, center=True).mean()
rain_mean = rain_mean.rolling(time=3, center=True).mean()

# ======================================================
# PLOT
# ======================================================
fig, ax = plt.subplots(figsize=(11, 5))

time = snow_mean["time"].values

# ensemble members
for i in range(snow_ice_ts.sizes["ens"]):
    ax.plot(time, snow_ice_ts.isel(ens=i), color="blue", alpha=0.08)

for i in range(rain_ice_ts.sizes["ens"]):
    ax.plot(time, rain_ice_ts.isel(ens=i), color="tab:orange", alpha=0.08)

# means
ax.plot(time, snow_mean, color="blue", linewidth=2.5, label="Snow-on-ice (daily mean)")
ax.plot(time, rain_mean, color="tab:orange", linewidth=2.5, label="Rain-on-ice (daily mean)")

# spread
ax.fill_between(time, snow_mean - snow_std, snow_mean + snow_std, color="blue", alpha=0.25)
ax.fill_between(time, rain_mean - rain_std, rain_mean + rain_std, color="tab:orange", alpha=0.25)

# labels
ax.set_ylabel("Mean Daily Fraction of Sea Ice Area Experiencing\nRain-on-Snow-on-Ice (ROSI) Event", size=12)
ax.set_xlabel("Year")

ax.legend(frameon=False)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("cesm2_snow_rain_on_ice_clean.png", dpi=300)
plt.show()

# Other Code

# Observational Data

In [ ]:
# ======================================================
# COMMON ANALYSIS PERIOD
# ======================================================
time_slice = slice("1980-08-01", "2021-07-31")

# ======================================================
# DIRECTORIES
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

OBS_ICE_FILES = ["/glade/work/xliu1/nasa_regrid/nasa_daily_1979_2000.nc", "/glade/work/xliu1/nasa_regrid/nasa_daily_2001_2024.nc"]

OBS_RAIN_FILES = ["/glade/work/xliu1/era5_precip_regrid/tp_1979-2000_128_143.nc", 
                  "/glade/work/xliu1/era5_precip_regrid/tp_2001-2024_128_143.nc"]

OBS_SNOW_FILE = "/glade/work/xliu1/era5_snow_regrid/SM_snod_ERA5_01Aug1980-31Jul2021_v01_1deg.nc"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005", 
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025"]

# ======================================================
# VARIABLES & THRESHOLDS
# ======================================================
ice_var  = "aice_d"
snow_var = "vsnon_d"
rain_var = "rain_d"

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh_obs   = 0.001
rain_thresh_model = 0.10

# ======================================================
# TIME CLEANER
# ======================================================
def fix_time_axis(da):
    da = da.sortby("time")
    _, index = np.unique(da["time"].values, return_index = True)
    da = da.isel(time = index)
    return da

# ================= OBSERVATIONS =======================
ice_obs = xr.open_mfdataset(OBS_ICE_FILES, combine = "nested", concat_dim = "time", chunks = {"time": 30})["cdr_seaice_conc"]
rain_obs = xr.open_mfdataset(OBS_RAIN_FILES, combine = "by_coords", chunks = {"time": 30})["tp"]
snow_obs = xr.open_dataset(OBS_SNOW_FILE, chunks = {"time": 30})["snod"]

ice_obs  = fix_time_axis(ice_obs)
rain_obs = fix_time_axis(rain_obs)
snow_obs = fix_time_axis(snow_obs)

ice_obs  = ice_obs.sel(time = time_slice)
rain_obs = rain_obs.sel(time = time_slice)
snow_obs = snow_obs.sel(time = time_slice)

ice_obs  = ice_obs.interp(lat = snow_obs.lat, lon = snow_obs.lon)
rain_obs = rain_obs.interp(lat = snow_obs.lat, lon = snow_obs.lon)

ice_obs, snow_obs, rain_obs = xr.align(ice_obs, snow_obs, rain_obs, join = "inner")

rosi_obs = ((ice_obs  >= ice_thresh) & (snow_obs >= snow_thresh) & (rain_obs >= rain_thresh_obs))
rosi_obs_freq = rosi_obs.mean("time")

# ==================== MODEL ===========================
def load_ros_part(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time": 30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time": 30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time": 30}) as ds_r:

        ice  = fix_time_axis(ds_i[ice_var])
        snow = fix_time_axis(ds_s[snow_var])
        rain = fix_time_axis(ds_r[rain_var])

        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ice  = ice.sel(time = time_slice)
        snow = snow.sel(time = time_slice)
        rain = rain.sel(time = time_slice)

        if ice.time.size == 0:
            return None

        ros = ((ice  >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh_model))

        return ros

model_maps = []

for ens in ensemble_members:

    print("Processing ensemble:", ens)

    ros_chunks = []

    for p in periods:

        ros = load_ros_part(f"{ICE_DIR}/aice_{p}_{ens}.nc", f"{SNOW_DIR}/snow_{p}_{ens}.nc", f"{RAIN_DIR}/rain_{p}_{ens}.nc")

        if ros is not None:
            ros_chunks.append(ros)

    if len(ros_chunks) == 0:
        continue

    ros_full = xr.concat(ros_chunks, dim = "time")
    ros_freq = ros_full.mean("time")

    model_maps.append(ros_freq)

model_maps = xr.concat(model_maps, dim = "ens")
model_mean = model_maps.mean("ens")

# ======================================================
# ALIGN & BIAS
# ======================================================
# Regrid model to obs grid
model_mean_interp = model_mean.interp(lat = rosi_obs_freq.lat, lon = rosi_obs_freq.lon)

bias = model_mean_interp - rosi_obs_freq

# ======================================================
# MAP SETTINGS
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

def setup_map(ax):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "lightgrey", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)
    return ax

# ======================================================
# DISCRETE COLOR LEVELS
# ======================================================
levels = [0, 0.001, 0.003, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]

cmap = plt.get_cmap("RdYlBu_r")
norm = BoundaryNorm(levels, cmap.N)

# ======================================================
# BIAS COLOR SETTINGS
# ======================================================
lim = 0.007

bias_levels = np.array([-lim, -0.005, -0.003, -0.001, 0, 0.001, 0.003, 0.005, lim])

n_bins = len(bias_levels) - 1
cmap_bias = plt.get_cmap("RdBu_r", n_bins)
norm_bias = BoundaryNorm(bias_levels, cmap_bias.N)

# ======================================================
# FIGURE LAYOUT
# ======================================================
fig = plt.figure(figsize = (18, 6))

gs = GridSpec(2, 4, figure = fig, height_ratios = [1, 0.05], width_ratios = [1, 1, 1, 0.05], wspace = 0.05)

ax1 = fig.add_subplot(gs[0, 0], projection = proj)
ax2 = fig.add_subplot(gs[0, 1], projection = proj)
ax3 = fig.add_subplot(gs[0, 2], projection = proj)

cax_bias = fig.add_subplot(gs[0, 3])
cax_bottom = fig.add_subplot(gs[1, 0:2])

setup_map(ax1)
setup_map(ax2)
setup_map(ax3)

# ------------------------------------------------------
# (a) Observations
# ------------------------------------------------------
p1 = ax1.pcolormesh(rosi_obs_freq.lon, rosi_obs_freq.lat, rosi_obs_freq, 
                    transform = transform, cmap = cmap, norm = norm, shading = "auto")

ax1.set_title("Observed Rain-on-Snow-on-Ice\n(ROSI) Frequency (1980–2021)", fontsize = 13)

ax1.text(0.02, 0.95, "(a)", transform = ax1.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# (b) Model
# ------------------------------------------------------
p2 = ax2.pcolormesh(model_mean_interp.lon, model_mean_interp.lat, model_mean_interp,
                    transform = transform, cmap = cmap, norm = norm, shading = "auto")

ax2.set_title("CESM2-LE Ensemble Mean\nROSI Frequency (1980–2021)", fontsize = 13)

ax2.text(0.02, 0.95, "(b)", transform = ax2.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# (c) Bias
# ------------------------------------------------------
p3 = ax3.pcolormesh(bias.lon, bias.lat, bias, transform = transform, cmap = cmap_bias, norm = norm_bias, shading = "auto")

ax3.set_title("CESM2-LE Bias (Model − Observations)", fontsize = 13)

ax3.text(0.02, 0.95, "(c)", transform = ax3.transAxes, fontsize = 14, fontweight = "bold",
         va = "top", ha = "left", bbox = dict(facecolor = "white", alpha = 0.7, edgecolor = "none"))

# ------------------------------------------------------
# Shared colorbar for OBS + MODEL
# ------------------------------------------------------
cbar = fig.colorbar(p2, cax = cax_bottom, orientation = "horizontal", extend = "max", boundaries = levels)

cbar.set_label("ROSI Frequency (fraction of days annually)", fontsize = 12)
cbar.set_ticks(levels)

# ------------------------------------------------------
# Vertical colorbar for BIAS
# ------------------------------------------------------
cbar_bias = fig.colorbar(p3, cax = cax_bias, orientation = "vertical", extend = "both", boundaries = bias_levels)

cbar_bias.set_label("ROSI Frequency Bias (model − observations)", fontsize = 12)
cbar_bias.set_ticks(bias_levels)
cbar_bias.ax.tick_params(labelsize = 10)

# ------------------------------------------------------
# Save
# ------------------------------------------------------
plt.savefig("ROSI_obs_model_bias_1980_2021.png", dpi = 300, bbox_inches = "tight")
plt.show()

## NASA Sea Ice (1979-2024)

In [ ]:
# Done - Confirmed
# FDR correction
try:
    from statsmodels.stats.multitest import fdrcorrection
    HAS_FDR = True
except Exception:
    HAS_FDR = False
    print("statsmodels.fdrcorrection not found — continuing without FDR correction")

# ------------------------
# USER PATHS / VAR
# ------------------------
nasa_early = "/glade/work/xliu1/nasa_regrid/nasa_daily_1979_2000.nc"
nasa_late  = "/glade/work/xliu1/nasa_regrid/nasa_daily_2001_2024.nc"
var = "cdr_seaice_conc"

# ------------------------
# LOAD
# ------------------------
ds1 = xr.open_dataset(nasa_early)
ds2 = xr.open_dataset(nasa_late)
nasa1 = ds1[var]
nasa2 = ds2[var]

# ------------------------
# PARAMETERS
# ------------------------
alpha = 0.05            # significance threshold for t-test
min_years = 8           # require at least this many independent years
edge_low = 0.15
edge_high = 0.85
stipple_density = 0.12
stipple_jitter = 0.25
stipple_seed = 42

# ------------------------
# 1) Annual fractions
# ------------------------
def annual_fraction(da):
    """Return DataArray: dims(time = year_end, lat, lon) = fraction of days with >= 0.30."""
    return (da >= 0.30).resample(time = "YE").mean("time")

ice_ann1 = annual_fraction(nasa1).compute()
ice_ann2 = annual_fraction(nasa2).compute()

# ------------------------
# 2) Basic checks
# ------------------------
ice_freq1 = (nasa1 >= 0.30).mean("time")
ice_freq2 = (nasa2 >= 0.30).mean("time")
ice_diff  = ice_freq2 - ice_freq1

spatial_dims = [d for d in ice_freq1.dims if d != "time"]
dim_y, dim_x = spatial_dims

# ------------------------
# 3) Minimum-year mask
# ------------------------
valid_counts1 = np.sum(~np.isnan(ice_ann1.values), axis = 0)
valid_counts2 = np.sum(~np.isnan(ice_ann2.values), axis = 0)
sufficient_samples = (valid_counts1 >= min_years) & (valid_counts2 >= min_years)

# ------------------------
# 4) Welch t-test
# ------------------------
tstat, pval = ttest_ind(ice_ann1.values, ice_ann2.values, axis = 0, equal_var = False, nan_policy = "omit")

pval_da = xr.DataArray(pval, coords = {dim_y: ice_freq1[dim_y], dim_x: ice_freq1[dim_x]}, dims = (dim_y, dim_x))

# set pvals to 1.0 where insufficient samples
pval_da = pval_da.where(sufficient_samples, other = 1.0)

# ------------------------
# 5) FDR correction
# ------------------------
if HAS_FDR:
    flat_p = pval_da.values.ravel()
    valid_flat = np.isfinite(flat_p)

    rejections = np.full_like(flat_p, fill_value = False, dtype = bool)

    if np.any(valid_flat):
        reject_flat, _ = fdrcorrection(flat_p[valid_flat], alpha = alpha, method = "indep")
        rejections[valid_flat] = reject_flat

    sig_stat = xr.DataArray(rejections.reshape(pval_da.shape), coords = pval_da.coords, dims = pval_da.dims)
else:
    sig_stat = pval_da < alpha

# ------------------------
# 6) New magnitude rule: decline magnitude ≥ 1.0 sigma
# ------------------------
# compute σ from early-period interannual variability
sigma1 = ice_ann1.std(dim = "time")  # (lat, lon)

# magnitude difference
abs_decline = (ice_freq1 - ice_freq2)

# require: decline ≥ 1σ
mag_mask = abs_decline >= sigma1

# keep only declines (freq2 < freq1)
decline_dir = (ice_freq2 < ice_freq1)

# combined magnitude mask
sigma_decline_mask = decline_dir & mag_mask

# ------------------------
# 7) Edge mask
# ------------------------
edge_mask = (ice_freq1 > edge_low) & (ice_freq1 < edge_high)

# final significance = stat sig + magnitude sig + edge filter
final_sig = sig_stat & sigma_decline_mask & edge_mask
final_sig = final_sig.fillna(False).astype(bool)

# ------------------------
# 8) Stippling (robust)
# ------------------------
lat = ice_freq1[dim_y]
lon = ice_freq1[dim_x]

def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):
    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)

    if lon_da.ndim == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 6)

# ------------------------
# 9) PLOTTING — two-period maps
# ------------------------
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

fig1, (ax1, ax2) = plt.subplots(ncols = 2, figsize = (12, 6), subplot_kw = dict(projection = proj), constrained_layout = True) 

cmap_main = plt.cm.cividis.copy()
cmap_main.set_bad("0.85")

p1 = ax1.pcolormesh(lon, lat, ice_freq1, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)
p2 = ax2.pcolormesh(lon, lat, ice_freq2, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)

# add stippling on late period
edge_mask_late = (ice_freq2 > 0.05) & (ice_freq2 < 0.60)
sig_edge = final_sig & edge_mask_late
add_stipple(ax2, lon, lat, sig_edge, density = stipple_density, jitter = stipple_jitter, 
            size = 6, alpha = 0.9, color = "tomato", seed = stipple_seed)

# shared formatting
for ax in (ax1, ax2):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax1.set_title("Fraction of days with SIC ≥30%\n1979–2000", fontsize = 13)
ax2.set_title("Fraction of days with SIC ≥30%\n2001–2024", fontsize = 13)

cbar = fig1.colorbar(p1, ax = [ax1, ax2], orientation = "vertical", shrink = 0.9, pad = 0.02)
cbar.ax.tick_params(labelsize = 10)
cbar.set_label("Fraction of days with sea-ice concentration ≥ 30%", fontsize = 12)

fig1.savefig("nasa_seaice_1.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ------------------------
# 10) Difference Map
# ------------------------
fig2 = plt.figure(figsize = (6, 6))
ax3 = fig2.add_subplot(1, 1, 1, projection = proj)

diff_min = float(ice_diff.min())
diff_max = float(ice_diff.max())
lim = max(abs(diff_min), abs(diff_max))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

p3 = ax3.pcolormesh(lon, lat, ice_diff, transform = transform, cmap = "RdBu_r", norm = norm)

ax3.set_extent(extent, crs = transform)
ax3.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
ax3.coastlines(linewidth = 0.9, color = "k", zorder = 6)
ax3.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax3.set_title("Change in sea-ice occurrence fraction\n(2001–2024 - 1979–2000)", fontsize = 12)

cbar2 = fig2.colorbar(p3, ax = ax3, orientation = "horizontal", shrink = 0.78, pad = 0.07)
cbar2.set_label("Δ fraction of days with SIC ≥30%", fontsize = 12)

fig2.savefig("nasa_seaice_2.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ------------------------
# Cleanup
# ------------------------
ds1.close()
ds2.close()

In [ ]:
# Done - Confirmed
# -------------------------------
# Merge both periods
# -------------------------------
# nasa is a DataArray (NOT a dataset), shape (time, dim_y, dim_x)
nasa = xr.concat([nasa1, nasa2], dim = "time")

# Mask for ≥ 30% concentration
ice_mask = (nasa >= 0.30)   # correct
       
# -------------------------------
# (1) Annual mean total ice-covered grid cells
# -------------------------------
ice_cells = ice_mask.sum(dim = (dim_y, dim_x))   # correct dims

annual_mean = ice_cells.groupby("time.year").mean()

plt.figure(figsize = (8, 4))
plt.plot(annual_mean["year"], annual_mean, color = "steelblue", linewidth = 2)
plt.title("Annual Mean Sea Ice Coverage — NASA (1979–2024)")
plt.xlabel("Year")
plt.ylabel("Mean number of ice-covered cells")
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

# -------------------------------
# (2) Daily climatological cycle
# -------------------------------
daily_cycle = (ice_mask.groupby("time.dayofyear").mean(dim = "time").sum(dim = (dim_y, dim_x)))

plt.figure(figsize = (8,4))
plt.plot(daily_cycle["dayofyear"], daily_cycle, color = "darkorange", linewidth = 2)
plt.title("Daily Mean Annual Cycle of Sea Ice Coverage — NASA (1979–2024)")
plt.xlabel("Day of Year")
plt.ylabel("Mean number of ice-covered cells")
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

## ERA5 Precipitation (1979-2024)

In [ ]:
# Done - Confirmed
# =========================
# FDR
# =========================
try:
    from statsmodels.stats.multitest import fdrcorrection
    HAS_FDR = True
except:
    HAS_FDR = False
    print("statsmodels not found — running without FDR")

# =========================
# FILES / VAR
# =========================
era5_early = "/glade/work/xliu1/era5_precip_regrid/tp_1979-2000_128_143.nc"
era5_late  = "/glade/work/xliu1/era5_precip_regrid/tp_2001-2024_128_143.nc"

VAR = "tp"
THRESH = 0.001    # 1 mm
alpha = 0.05
min_years = 10

edge_low  = 0.05  
edge_high = 0.95

stipple_density = 0.12
stipple_jitter  = 0.25
stipple_seed    = 42

# =========================
# LOAD
# =========================
ds1 = xr.open_dataset(era5_early, chunks = {"time": 30})
ds2 = xr.open_dataset(era5_late,  chunks = {"time": 30})

pr1 = ds1[VAR]
pr2 = ds2[VAR]

lat = pr1.lat
lon = pr1.lon

# =========================
# 1) ANNUAL FRACTIONS
# =========================
def annual_fraction(da):
    return (da >= THRESH).resample(time = "YE").mean("time")

ann1 = annual_fraction(pr1).compute()
ann2 = annual_fraction(pr2).compute()

# =========================
# 2) MEAN FRACTIONS
# =========================
freq1 = (pr1 >= THRESH).mean("time")
freq2 = (pr2 >= THRESH).mean("time")
diff  = freq2 - freq1

spatial_dims = [d for d in freq1.dims if d != "time"]
dim_y, dim_x = spatial_dims

# =========================
# 3) MINIMUM SAMPLE MASK
# =========================
n1 = np.sum(~np.isnan(ann1.values), axis = 0)
n2 = np.sum(~np.isnan(ann2.values), axis = 0)
sufficient = (n1 >= min_years) & (n2 >= min_years)

# =========================
# 4) WELCH T-TEST
# =========================
tstat, pval = ttest_ind(ann1.values, ann2.values, axis = 0, equal_var = False, nan_policy = "omit")

pval_da = xr.DataArray(pval, coords = {dim_y: freq1[dim_y], dim_x: freq1[dim_x]}, dims = (dim_y, dim_x))

pval_da = pval_da.where(sufficient, other = 1.0)

# =========================
# 5) FDR CORRECTION
# =========================
if HAS_FDR:
    flat = pval_da.values.ravel()
    valid = np.isfinite(flat)
    reject = np.zeros_like(flat, dtype = bool)

    if valid.any():
        r, _ = fdrcorrection(flat[valid], alpha = alpha, method = "indep")
        reject[valid] = r

    sig_stat = xr.DataArray(reject.reshape(pval_da.shape), coords = pval_da.coords, dims = pval_da.dims)
else:
    sig_stat = pval_da < alpha

# =========================
# 6) MAGNITUDE FILTER (≥1σ)
# =========================
sigma1 = ann1.std(dim = "time")
abs_change = np.abs(freq2 - freq1)
mag_mask = abs_change >= sigma1

# =========================
# 7) EDGE MASK
# =========================
edge_mask = (freq1 > edge_low) & (freq1 < edge_high)

# =========================
# 8) FINAL SIGNIFICANCE
# =========================
final_sig = sig_stat & mag_mask & edge_mask
final_sig = final_sig.fillna(False)

# =========================
# STIPPLE FUNCTION (robust)
# =========================
def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.25, size = 6, alpha = 0.9, color = "tomato", seed = 42):

    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)

    if lon_da.ndim == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 6)

# =========================
# MAP SETUP
# =========================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

# =========================
# FIGURE 1 — MEAN MAPS
# =========================
fig1, (ax1, ax2) = plt.subplots(ncols = 2, figsize = (12, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

cmap_main = plt.cm.cividis.copy()
cmap_main.set_bad("0.85")

p1 = ax1.pcolormesh(lon, lat, freq1, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)
p2 = ax2.pcolormesh(lon, lat, freq2, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)

for ax in (ax1, ax2):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax1.set_title("Fraction of days with precipitation ≥1mm day⁻¹\n1979–2000", fontsize = 13)
ax2.set_title("Fraction of days with precipitation ≥1mm day⁻¹\n2001–2024", fontsize = 13)

# stipple only on late-period edge
edge_late = (freq2 > edge_low) & (freq2 < edge_high)
add_stipple(ax2, lon, lat, final_sig & edge_late, density = stipple_density, jitter = stipple_jitter, seed = stipple_seed)

cbar = fig1.colorbar(p1, ax = [ax1, ax2], orientation = "vertical", shrink = 0.9, pad = 0.02)
cbar.ax.tick_params(labelsize = 10)
cbar.set_label("Fraction of days with precipitation ≥1mm day⁻¹", fontsize = 12)

fig1.savefig("era5_rain_fraction.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =========================
# FIGURE 2 — DIFFERENCE MAP
# =========================
fig2 = plt.figure(figsize = (6, 6))
ax = fig2.add_subplot(1, 1, 1, projection = proj)

diff_min = float(diff.min())
diff_max = float(diff.max())
lim = max(abs(diff_min), abs(diff_max))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

p = ax.pcolormesh(lon, lat, diff, transform = transform, cmap = "RdBu_r", norm = norm)

ax.set_extent(extent, crs = transform)
ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax.set_title("Change in precipitation-day fraction\n(2001–2024 - 1979–2000)", fontsize = 12)

cbar2 = fig2.colorbar(p, ax = ax, orientation = "horizontal", shrink = 0.78, pad = 0.07)
cbar2.set_label("Δ fraction of days with precipiation ≥1mm day⁻¹", fontsize = 11)

fig2.savefig("era5_rain_difference.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =========================
# CLEANUP
# =========================
ds1.close()
ds2.close()

## ERA5 Snow Depth (1980-2021)

In [ ]:
# Done - Confirmed
# =========================
# FDR
# =========================
try:
    from statsmodels.stats.multitest import fdrcorrection
    HAS_FDR = True
except:
    HAS_FDR = False
    print("statsmodels not found — running without FDR")

# =========================
# FILE / VAR
# =========================
FILE = "/glade/work/xliu1/era5_snow_regrid/SM_snod_ERA5_01Aug1980-31Jul2021_v01_1deg.nc"
VAR = "snod"       # snow depth (m)

THRESH = 0.1 
alpha = 0.05
min_years = 10

# edge definition
edge_low  = 0.05
edge_high = 0.95

stipple_density = 0.12
stipple_jitter  = 0.25
stipple_seed    = 42

# =========================
# LOAD
# =========================
ds = xr.open_dataset(FILE, chunks = {"time": 30})
snow = ds[VAR]

lat = snow.lat
lon = snow.lon

# =========================
# SPLIT PERIODS
# =========================
snow1 = snow.sel(time = slice("1980-01-01", "2000-12-31"))
snow2 = snow.sel(time = slice("2001-01-01", "2021-07-31"))

# =========================
# ANNUAL FRACTIONS
# =========================
def annual_fraction(da):
    return (da >= THRESH).resample(time = "YE").mean("time")

ann1 = annual_fraction(snow1).compute()
ann2 = annual_fraction(snow2).compute()

# =========================
# MEAN FRACTIONS
# =========================
freq1 = (snow1 >= THRESH).mean("time")
freq2 = (snow2 >= THRESH).mean("time")
diff  = freq2 - freq1

spatial_dims = [d for d in freq1.dims if d != "time"]
dim_y, dim_x = spatial_dims

# =========================
# MINIMUM SAMPLE FILTER
# =========================
valid_counts1 = np.sum(~np.isnan(ann1.values), axis = 0)
valid_counts2 = np.sum(~np.isnan(ann2.values), axis = 0)
sufficient_samples = (valid_counts1 >= min_years) & (valid_counts2 >= min_years)

# =========================
# T-TEST
# =========================
tstat, pval = ttest_ind(ann1.values, ann2.values, axis = 0, equal_var = False, nan_policy = "omit")

pval_da = xr.DataArray(pval, coords = {dim_y: freq1[dim_y], dim_x: freq1[dim_x]}, dims = (dim_y, dim_x))
pval_da = pval_da.where(sufficient_samples, other = 1.0)

# =========================
# FDR
# =========================
if HAS_FDR:
    flat = pval_da.values.ravel()
    valid = np.isfinite(flat)
    reject = np.zeros_like(flat, dtype = bool)

    if valid.any():
        r, _ = fdrcorrection(flat[valid], alpha = alpha)
        reject[valid] = r

    sig_stat = xr.DataArray(reject.reshape(pval_da.shape), coords = pval_da.coords, dims = pval_da.dims)
else:
    sig_stat = pval_da < alpha

# =========================
# MAGNITUDE FILTER (≥1σ)
# =========================
sigma1 = ann1.std(dim = "time")
abs_change = np.abs(freq2 - freq1)
mag_mask = abs_change >= sigma1

# =========================
# EDGE MASK (remove permanent/never snow)
# =========================
edge_mask = (freq1 > edge_low) & (freq1 < edge_high)

# =========================
# FINAL SIGNIFICANCE
# =========================
final_sig = sig_stat & mag_mask & edge_mask
final_sig = final_sig.fillna(False)

# =========================
# STIPPLE FUNCTION (robust)
# =========================
def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.25, size = 6, alpha = 0.9, color = "tomato", seed = 42):
    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)

    if lon_da.ndim == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 6)

# =========================
# MAP SETUP
# =========================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

# =========================
# FIGURE 1 — MEANS
# =========================
fig1, (ax1, ax2) = plt.subplots(ncols = 2, figsize = (12, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

cmap_main = plt.cm.cividis.copy()
cmap_main.set_bad("0.85")

p1 = ax1.pcolormesh(lon, lat, freq1, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)
p2 = ax2.pcolormesh(lon, lat, freq2, transform = transform, cmap = cmap_main, vmin = 0, vmax = 1)

for ax in (ax1, ax2):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax1.set_title("Fraction of days with snow depth ≥0.1m day⁻¹\n1980–2000", fontsize = 13)
ax2.set_title("Fraction of days with snow depth ≥0.1m day⁻¹\n2001–2021", fontsize = 13)

add_stipple(ax2, lon, lat, final_sig, density = stipple_density, jitter = stipple_jitter, seed = stipple_seed)

cbar = fig1.colorbar(p1, ax = [ax1, ax2], orientation = "vertical", shrink = 0.9, pad = 0.02)
cbar.set_label("Fraction of days with snow depth ≥0.1m day⁻¹", fontsize = 12)

fig1.savefig("era5_snow_fraction.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =========================
# FIGURE 2 — DIFFERENCE
# =========================
fig2 = plt.figure(figsize = (6, 6))
ax = fig2.add_subplot(1, 1, 1, projection = proj)

diff_min = float(diff.min())
diff_max = float(diff.max())
lim = max(abs(diff_min), abs(diff_max))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

p = ax.pcolormesh(lon, lat, diff, transform = transform, cmap = "RdBu_r", norm = norm)

ax.set_extent(extent, crs = transform)
ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

ax.set_title("Change in snow cover-day fraction\n(2001–2021 - 1980–2000)", fontsize = 12)

cbar2 = fig2.colorbar(p, ax = ax, orientation = "horizontal", shrink = 0.78, pad = 0.07)
cbar2.set_label("Δ fraction of days with snow depth ≥0.1m day⁻¹", fontsize = 11)

fig2.savefig("era5_snow_difference.png", dpi = 300, bbox_inches = "tight")
plt.show()

ds.close()

# Code for CESM2 All

## Sea Ice 1970-2100

In [ ]:
# Done - Confirmed
def run_seaice_pipeline(ens_tag, basedir):
    """
    Run full sea ice analysis & plotting for one ensemble member.
    ens_tag e.g. "1011.001"
    """

    VERIFY_PLOTS = True   # set False for full batch runs

    # Optional: FDR correction
    try:
        from statsmodels.stats.multitest import fdrcorrection
        HAS_FDR = True
    except Exception:
        HAS_FDR = False
        print("statsmodels.fdrcorrection not found — continuing without FDR correction")

    # ------------------------
    # FILES
    # ------------------------
    seaice_me = f"{basedir}/aice_1970-2000_{ens_tag}.nc"
    seaice_mr = f"{basedir}/aice_2000-2025_{ens_tag}.nc"
    seaice_fe = f"{basedir}/aice_2025-2065_{ens_tag}.nc"
    seaice_fl = f"{basedir}/aice_2065-2100_{ens_tag}.nc"

    print(f"\n=== Running ensemble {ens_tag} ===")

    # ------------------------
    # LOAD DATA
    # ------------------------
    ds1 = xr.open_dataset(seaice_me, chunks = {"time": 30})
    ds2 = xr.open_dataset(seaice_mr, chunks = {"time": 30})
    ds3 = xr.open_dataset(seaice_fe, chunks = {"time": 30})
    ds4 = xr.open_dataset(seaice_fl, chunks = {"time": 30})

    var = "aice_d"  # sea ice concentration (0–1)

    seaice1 = ds1[var]
    seaice2 = ds2[var]
    seaice3 = ds3[var]
    seaice4 = ds4[var]

    # ---- lon/lat detection (unchanged) ----
    if ("lon" in seaice1.coords) and ("lat" in seaice1.coords):
        lon = seaice1["lon"]
        lat = seaice1["lat"]
    else:
        spatial = [d for d in seaice1.dims if d != "time"]
        lat = seaice1[spatial[0]]
        lon = seaice1[spatial[1]]

    # ------------------------
    # PARAMETERS
    # ------------------------
    alpha = 0.05          # p-value threshold
    min_years = 8         # minimum independent years required in each period
    edge_low = 0.15
    edge_high = 0.85
    stipple_density = 0.12
    stipple_jitter = 0.25
    stipple_seed = 42

    # ------------------------
    # Annual fractions
    # ------------------------
    # ------------------------
    # 1) Annual fractions (YE = year-end)
    # ------------------------
    def annual_fraction(da):
        """Return DataArray with dims (time = years, y/x) = fraction of days in year with >=0.30"""
        return (da >= 0.30).resample(time = "YE").mean("time")
        
    ann1 = annual_fraction(seaice1).compute()
    ann2 = annual_fraction(seaice2).compute()
    ann3 = annual_fraction(seaice3).compute()
    ann4 = annual_fraction(seaice4).compute()

    # ------------------------
    # Mean fractions
    # ------------------------
    ice_frac1 = (seaice1 >= 0.30).mean(dim = "time")
    ice_frac2 = (seaice2 >= 0.30).mean(dim = "time")
    ice_frac3 = (seaice3 >= 0.30).mean(dim = "time")
    ice_frac4 = (seaice4 >= 0.30).mean(dim = "time")

    diff_2025v1970 = ice_frac2 - ice_frac1
    diff_2100v2025 = ice_frac4 - ice_frac3
    diff_2100v1970 = ice_frac4 - ice_frac1

    # detect spatial dim names
    spatial_dims = [d for d in ice_frac1.dims if d != "time"]
    if len(spatial_dims) != 2:
        raise RuntimeError(f"Expected 2 spatial dims, found {spatial_dims}")
    dim_y, dim_x = spatial_dims

    # ------------------------
    # Significance masks
    # ------------------------

    def significance_mask(annA, annB, iceA, iceB, alpha = 0.05, min_years = 8, use_fdr = HAS_FDR):
        """
        annA, annB : annual DataArray (time, y, x)
        iceA, iceB : mean-fraction DataArray (y, x) for the same periods
        returns: boolean DataArray (y, x) of final significance (stat + magnitude + edge)
        """
        # 1) minimum years
        countsA = np.sum(~np.isnan(annA.values), axis = 0)
        countsB = np.sum(~np.isnan(annB.values), axis = 0)
        sufficient = (countsA >= min_years) & (countsB >= min_years)
    
        # 2) Welch t-test (naive call, nan_policy = 'omit')
        tstat, pval = ttest_ind(annA.values, annB.values, axis = 0, equal_var = False, nan_policy = "omit")
        pval_da = xr.DataArray(pval, coords = {dim_y: iceA[dim_y], dim_x: iceA[dim_x]}, dims = (dim_y, dim_x))
        pval_da = pval_da.where(sufficient, other = 1.0)
    
        # 3) Multiple testing: FDR if available, else raw p < alpha
        if use_fdr:
            flatp = pval_da.values.ravel()
            valid = np.isfinite(flatp)
            reject = np.full_like(flatp, False, dtype=bool)
            if valid.any():
                reject_valid, _ = fdrcorrection(flatp[valid], alpha = alpha, method = "indep")
                reject[valid] = reject_valid
            stat_sig = xr.DataArray(reject.reshape(pval_da.shape), coords = pval_da.coords, dims = pval_da.dims)
        else:
            stat_sig = pval_da < alpha
    
        # 4) magnitude requirement: decline >= sigma (use annA as baseline)
        sigmaA = annA.std(dim = "time")
        decline = iceA - iceB  # positive when ice decreased from A->B
        mag_mask = decline >= sigmaA
    
        # 5) only declines
        decline_dir = iceB < iceA
    
        # 6) seasonal-edge region (avoid permanent pack/open ocean)
        edge_mask = (iceA > edge_low) & (iceA < edge_high)
    
        # 7) final mask
        final = stat_sig & mag_mask & decline_dir & edge_mask
        final = final.fillna(False).astype(bool)
        return final, pval_da, sigmaA
    
    sig_2025v1970, _, _ = significance_mask(ann1, ann2, ice_frac1, ice_frac2, alpha = alpha, min_years = min_years)
    sig_2065v2000, _, _ = significance_mask(ann2, ann3, ice_frac2, ice_frac3, alpha = alpha, min_years = min_years)
    sig_2100v2025, _, _ = significance_mask(ann3, ann4, ice_frac3, ice_frac4, alpha = alpha, min_years = min_years)

    # ------------------------
    # Robust stippling function (1D or 2D lon/lat)
    # ------------------------
    def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):
        """
        mask_da: xarray boolean with dims (dim_y, dim_x)
        lon_da, lat_da: either 1D or 2D arrays
        """
        np.random.seed(seed)
        ys, xs = np.where(mask_da.values)
        if len(xs) == 0:
            return
        if getattr(lon_da, "ndim", None) == 2:
            sig_lon = lon_da.values[ys, xs]
            sig_lat = lat_da.values[ys, xs]
        else:
            sig_lon = lon_da.values[xs]
            sig_lat = lat_da.values[ys]
    
        # thin
        keep = np.random.rand(len(sig_lon)) < density
        if keep.sum() == 0:
            return
        sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
        sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    
        ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, 
                   transform = ccrs.PlateCarree(), zorder = 6)
    
    # ------------------------
    # Plotting: keep your original layout but use new masks
    # ------------------------
    proj = ccrs.NorthPolarStereo()
    transform = ccrs.PlateCarree()
    extent = [-180, 180, 50, 90]
    
    def setup_map(ax):
        ax.set_extent(extent, crs = transform)
        ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
        ax.coastlines(linewidth = 0.9, color = "k", zorder = 6)
        gl = ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8, linestyle = "-")
        return ax

    # ------------------------
    # FIGURE 1 — Mean maps
    # ------------------------
    fig1, axs = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

    maps = [ice_frac1, ice_frac2, ice_frac3, ice_frac4]
    titles = [
        f"Fraction ≥30% Sea Ice (1970–2000)\nEns {ens_tag}", "Fraction ≥30% Sea Ice (2000–2025)",
        "Fraction ≥30% Sea Ice (2025–2065)", "Fraction ≥30% Sea Ice (2065–2100)",]
    stip_masks = [None, sig_2025v1970, sig_2065v2000, sig_2100v2025]

    for ax, data, title, mask in zip(axs.flat, maps, titles, stip_masks):
        setup_map(ax)
        p = ax.pcolormesh(lon, lat, data, transform = transform, cmap = plt.cm.cividis, vmin = 0, vmax = 1, shading = "auto")
        ax.set_title(title, fontsize = 13)

        if mask is not None:
            edge_mask_plot = (data > 0.05) & (data < 0.60)
            add_stipple(ax, lon, lat, mask & edge_mask_plot, density = stipple_density, 
                        jitter = stipple_jitter, seed = stipple_seed)

    cbar = fig1.colorbar(p, ax = axs.ravel().tolist(), shrink = 0.78, pad = 0.02)
    cbar.set_label("Fraction of Days ≥30% Ice")
    
    # fig1.savefig(f"cesm2_seaice_fraction_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()

    plt.close(fig1)

    # ------------------------
    # FIGURE 2 — Difference maps
    # ------------------------
    diffs = [diff_2025v1970, diff_2100v2025, diff_2100v1970]
    titles_diff = ["2000–2025 minus 1970–2000", "2065–2100 minus 2025–2065", "2065–2100 minus 1970–2000"]

    vmin = min(float(d.min()) for d in diffs)
    vmax = max(float(d.max()) for d in diffs)
    lim = max(abs(vmin), abs(vmax))
    norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

    fig2, axs2 = plt.subplots(
        ncols = 3, figsize = (18, 6),
        subplot_kw = dict(projection = proj),
        constrained_layout = True
    )

    for ax, data, title in zip(axs2, diffs, titles_diff):
        setup_map(ax)
        p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")
        ax.set_title(title)

    cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.08)
    cbar2.set_label("Change in Fraction ≥30% Ice")

    fig2.savefig(f"cesm2_seaice_diff_{ens_tag}.png", dpi = 300, bbox_inches = "tight")
    
    if VERIFY_PLOTS:
        plt.show()
    
    plt.close(fig2)

basedir = "/glade/work/xliu1/cesm2_seaice_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005", 
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

for ens in ensemble_members:
    run_seaice_pipeline(ens, basedir)

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# -----------------------------
# SETTINGS
# -----------------------------
basedir = "/glade/work/xliu1/cesm2_seaice_regrid"
ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

ice_threshold = 0.30
snr_threshold = 1.0

# -----------------------------
# LOAD FUNCTION
# -----------------------------
def annual_fraction(file):
    ds = xr.open_dataset(file, chunks = {"time":30})
    da = ds["aice_d"]

    ice = (da >= ice_threshold)

    # keep variability: fraction per year
    yearly = ice.groupby("time.year").mean("time")

    return yearly

# -----------------------------
# LOAD ALL ENSEMBLES
# -----------------------------
frac1_all, frac2_all, frac3_all, frac4_all = [], [], [], []

for ens in ensemble_members:
    print(f"Loading ensemble {ens}")
    frac1_all.append(annual_fraction(f"{basedir}/aice_1970-2000_{ens}.nc"))
    frac2_all.append(annual_fraction(f"{basedir}/aice_2000-2025_{ens}.nc"))
    frac3_all.append(annual_fraction(f"{basedir}/aice_2025-2065_{ens}.nc"))
    frac4_all.append(annual_fraction(f"{basedir}/aice_2065-2100_{ens}.nc"))

frac1_all = xr.concat(frac1_all, dim = "ens").assign_coords(ens = ensemble_members)
frac2_all = xr.concat(frac2_all, dim = "ens").assign_coords(ens = ensemble_members)
frac3_all = xr.concat(frac3_all, dim = "ens").assign_coords(ens = ensemble_members)
frac4_all = xr.concat(frac4_all, dim = "ens").assign_coords(ens = ensemble_members)

lon = frac1_all.lon
lat = frac1_all.lat

# =====================================================
# FIGURE 1 — ENSEMBLE MEAN STATE
# =====================================================
def collapse_state(frac):
    # remove BOTH ensemble + year dimensions safely
    return frac.mean(dim = [d for d in frac.dims if d in ["ens", "year"]])

mean1 = collapse_state(frac1_all)
mean2 = collapse_state(frac2_all)
mean3 = collapse_state(frac3_all)
mean4 = collapse_state(frac4_all)

print("Collapsed dims:", mean1.dims, mean1.shape)

proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

fig1, axs1 = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

state_maps = [mean1, mean2, mean3, mean4]
state_titles = ["1970–2000 Ensemble Mean", "2000–2025 Ensemble Mean", "2025–2065 Ensemble Mean", "2065–2100 Ensemble Mean"]

for ax, data, title in zip(axs1.flat, state_maps, state_titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p1 = ax.pcolormesh(data.lon, data.lat, data, transform = transform, cmap = "cividis",
                       vmin = 0, vmax = 1, shading = "nearest", zorder = 1)

    ax.set_title(title)

cbar1 = fig1.colorbar(p1, ax = axs1.ravel().tolist(), pad = 0.02)
cbar1.set_label("Fraction of Days ≥30% Ice")
fig1.savefig("cesm2_seaice_mean_state.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 2 — ENSEMBLE MEAN CHANGE + SNR STIPPLING
# =====================================================
landmask = xr.where(frac1_all.isel(ens = 0).notnull(), 1, np.nan)

# -----------------------------------------------------
# PERIOD MEANS PER ENSEMBLE MEMBER
# (needed for proper internal variability estimate)
# -----------------------------------------------------
mean1_ens = frac1_all.mean("year")
mean2_ens = frac2_all.mean("year")
mean3_ens = frac3_all.mean("year")
mean4_ens = frac4_all.mean("year")

# -----------------------------------------------------
# SIGNAL = ensemble-mean change
# -----------------------------------------------------
signal1 = mean2_ens.mean("ens") - mean1_ens.mean("ens")
signal2 = mean4_ens.mean("ens") - mean3_ens.mean("ens")
signal3 = mean4_ens.mean("ens") - mean1_ens.mean("ens")

# -----------------------------------------------------
# NOISE = ensemble spread of the change
# -----------------------------------------------------
noise1 = (mean2_ens - mean1_ens).std("ens")
noise2 = (mean4_ens - mean3_ens).std("ens")
noise3 = (mean4_ens - mean1_ens).std("ens")

eps = 1e-6   # tiny threshold to avoid divide-by-zero

noise1_safe = noise1.where(noise1 > eps)
noise2_safe = noise2.where(noise2 > eps)
noise3_safe = noise3.where(noise3 > eps)

snr1 = signal1 / noise1_safe
snr2 = signal2 / noise2_safe
snr3 = signal3 / noise3_safe

mask1 = np.abs(snr1) > snr_threshold
mask2 = np.abs(snr2) > snr_threshold
mask3 = np.abs(snr3) > snr_threshold

print("SNR1 range:", float(snr1.min().compute()), float(snr1.max().compute()))
print("SNR2 range:", float(snr2.min().compute()), float(snr2.max().compute()))
print("SNR3 range:", float(snr3.min().compute()), float(snr3.max().compute()))

print("\nNoise1 min/max:", float(noise1.min().compute()), float(noise1.max().compute()))
print("Noise2 min/max:", float(noise2.min().compute()), float(noise2.max().compute()))
print("Noise3 min/max:", float(noise3.min().compute()), float(noise3.max().compute()))

print("\nSignal1 min/max:", float(signal1.min().compute()), float(signal1.max().compute()))
print("Signal2 min/max:", float(signal2.min().compute()), float(signal2.max().compute()))
print("Signal3 min/max:", float(signal3.min().compute()), float(signal3.max().compute()))

# -----------------------------
# SELECT WHICH SIGNAL TO PLOT
# -----------------------------
sig = signal3 

vals_all = sig.data.compute().ravel()
vals_all = vals_all[np.isfinite(vals_all)]

baseline = mean1_ens.mean("ens") if "ens" in mean1_ens.dims else mean1
ice_mask = baseline > 0.15

vals_ice = sig.where(ice_mask).data.compute().ravel()
vals_ice = vals_ice[np.isfinite(vals_ice)]

# -----------------------------
# Histogram Plot
# -----------------------------
fig, axs = plt.subplots(2, 1, figsize = (8, 8), constrained_layout = True)

# --- ALL CELLS ---
axs[0].hist(vals_all, bins = 80)
axs[0].axvline(0, linestyle = "--")
axs[0].set_title("Change distribution — ALL grid cells")
axs[0].set_xlabel("Δ Fraction ≥30% Ice")
axs[0].set_ylabel("Count")

# --- ICE CELLS ONLY ---
axs[1].hist(vals_ice, bins = 80)
axs[1].axvline(0, linestyle = "--")
axs[1].set_title("Change distribution — ice-covered cells (baseline ≥15%)")
axs[1].set_xlabel("Δ Fraction ≥30% Ice")
axs[1].set_ylabel("Count")

plt.savefig("cesm2_seaice_histograms.png", dpi = 300, bbox_inches = "tight")
plt.show()

def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):

    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)
    if len(xs) == 0:
        return

    if getattr(lon_da, "ndim", None) == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 30)

fig2, axs2 = plt.subplots(1, 3, figsize = (18,6), subplot_kw = dict(projection = proj), constrained_layout = True)

diff_maps = [signal1, signal2, signal3]
diff_masks = [mask1, mask2, mask3]
diff_titles = ["2000–2025 minus 1970–2000", "2065–2100 minus 2025–2065", "2065–2100 minus 1970–2000"]

vmin = min(float(m.min().compute()) for m in diff_maps)
vmax = max(float(m.max().compute()) for m in diff_maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, mask, title in zip(axs2, diff_maps, diff_masks, diff_titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto", zorder = 1)

    add_stipple(ax, lon, lat, mask, density = 0.05, size = 4, alpha = 0.7)
    ax.set_title(title)

cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.07)
cbar2.set_label("Change in Fraction ≥30% Ice")

fig2.savefig("cesm2_seaice_mean_change.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 3 — SIGNAL MAPS
# =====================================================
fig3, axs3 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

signal_maps = [signal1, signal2, signal3]
titles_signal = ["Signal: 2000–2025 − 1970–2000", "Signal: 2065–2100 − 2025–2065", "Signal: 2065–2100 − 1970–2000"]

vmin = min(float(m.min().compute()) for m in signal_maps)
vmax = max(float(m.max().compute()) for m in signal_maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, title in zip(axs3, signal_maps, titles_signal):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)
    p3 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto", zorder = 1)
    ax.set_title(title)

cbar3 = fig3.colorbar(p3, ax = axs3, orientation = "horizontal", pad = 0.07)
cbar3.set_label("Signal (Ensemble Mean Change)")
fig3.savefig("cesm2_seaice_signal.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 4 — NOISE MAPS
# =====================================================
fig4, axs4 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

noise_maps = [noise1, noise2, noise3]
titles_noise = ["Noise: 2000–2025 − 1970–2000", "Noise: 2065–2100 − 2025–2065", "Noise: 2065–2100 − 1970–2000"]

vmax_noise = max(float(m.max().compute()) for m in noise_maps)

for ax, data, title in zip(axs4, noise_maps, titles_noise):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)
    p4 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "magma", 
                       vmin = 0, vmax = vmax_noise, shading = "auto", zorder = 1)
    ax.set_title(title)

cbar4 = fig4.colorbar(p4, ax = axs4, orientation = "horizontal", pad = 0.07)
cbar4.set_label("Internal Variability (σ across ensemble)")
fig4.savefig("cesm2_seaice_noise.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 5 — SNR MAPS (ACTUAL VALUES)
# =====================================================
fig5, axs5 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

snr_maps = [snr1, snr2, snr3]
titles_snr = ["SNR: 2000–2025 − 1970–2000", "SNR: 2065–2100 − 2025–2065", "SNR: 2065–2100 − 1970–2000"]

# symmetric color scale centered at 0
vmin = min(float(m.min().compute()) for m in snr_maps)
vmax = max(float(m.max().compute()) for m in snr_maps)

lim = min(max(abs(vmin), abs(vmax)), 5)

norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, title in zip(axs5, snr_maps, titles_snr):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p5 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto", zorder = 1)

    ax.set_title(title)

cbar5 = fig5.colorbar(p5, ax = axs5, orientation = "horizontal", pad = 0.07)
cbar5.set_label("Signal-to-Noise Ratio (μ change / σ ensemble)")

fig5.savefig("cesm2_seaice_snr.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
# Done - Confirmed
# ------------------------
# SETTINGS
# ------------------------
basedir = "/glade/work/xliu1/cesm2_seaice_regrid"
ens = "1011.001"
var = "aice_d"   # sea ice concentration
threshold = 0.30

# ------------------------
# LOAD DATA
# ------------------------
ds1 = xr.open_dataset(f"{basedir}/aice_1970-2000_{ens}.nc", chunks = {"time": 30})
ds2 = xr.open_dataset(f"{basedir}/aice_2000-2025_{ens}.nc", chunks = {"time": 30})
ds3 = xr.open_dataset(f"{basedir}/aice_2025-2065_{ens}.nc", chunks = {"time": 30})
ds4 = xr.open_dataset(f"{basedir}/aice_2065-2100_{ens}.nc", chunks = {"time": 30})

seaice1 = ds1[var]
seaice2 = ds2[var]
seaice3 = ds3[var]
seaice4 = ds4[var]

# ------------------------
# Detect spatial dimensions robustly
# ------------------------
spatial_dims = [d for d in seaice1.dims if d != "time"]
if len(spatial_dims) != 2:
    raise RuntimeError(f"Expected 2 spatial dims, found {spatial_dims}")

# ------------------------
# Boolean mask for ≥30% sea ice
# ------------------------
ice_mask1 = seaice1 >= threshold
ice_mask2 = seaice2 >= threshold
ice_mask3 = seaice3 >= threshold
ice_mask4 = seaice4 >= threshold

# ------------------------
# Count ice-covered grid cells per day
# ------------------------
ice_cells1 = ice_mask1.sum(dim = spatial_dims)
ice_cells2 = ice_mask2.sum(dim = spatial_dims)
ice_cells3 = ice_mask3.sum(dim = spatial_dims)
ice_cells4 = ice_mask4.sum(dim = spatial_dims)

# ------------------------
# Annual mean grid-cell counts
# ------------------------
annual_mean1 = ice_cells1.groupby("time.year").mean()
annual_mean2 = ice_cells2.groupby("time.year").mean()
annual_mean3 = ice_cells3.groupby("time.year").mean()
annual_mean4 = ice_cells4.groupby("time.year").mean()

# ------------------------
# Helper function: remove edge years (spikes)
# ------------------------
def clean_edges(annual_mean):
    years = annual_mean["year"].values
    if len(years) > 2:
        return annual_mean.sel(year = slice(years[1], years[-2]))
    else:
        return annual_mean

annual_mean1 = clean_edges(annual_mean1)
annual_mean2 = clean_edges(annual_mean2)
annual_mean3 = clean_edges(annual_mean3)
annual_mean4 = clean_edges(annual_mean4)

# ------------------------
# Plot
# ------------------------
plt.figure(figsize = (10, 5))

plt.plot(annual_mean1["year"], annual_mean1, label = "1970–2000", color = "steelblue")
plt.plot(annual_mean2["year"], annual_mean2, label = "2000–2025", color = "darkorange")
plt.plot(annual_mean3["year"], annual_mean3, label = "2025–2065", color = "forestgreen")
plt.plot(annual_mean4["year"], annual_mean4, label = "2065–2100", color = "firebrick")

plt.ylabel("Mean number of ice-covered grid cells (annual mean)")
plt.xlabel("Year")
plt.title(f"Annual Mean Sea Ice Coverage (≥30%)\nCESM2 Ensemble {ens}")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

# ------------------------
# Clean up
# ------------------------
ds1.close()
ds2.close()
ds3.close()
ds4.close()

In [ ]:
# Done - Confirmed
# ============================================================
# Single-ensemble sea ice diagnostics (daily + climatology)
# Ensemble: 1011.001
# ============================================================

ens_tag = "1011.001"

# ------------------------------------------------------------
# Helper: convert CFTime → datetime64
# ------------------------------------------------------------
def clean_time(da):
    da = da.copy()
    if "cftime" in str(type(da.time.values[0])).lower():
        da["time"] = da.indexes["time"].to_datetimeindex()
    return da.compute()

# ------------------------------------------------------------
# Collect ice-cell time series for ONE ensemble
# ------------------------------------------------------------
ice_cells = {"1970–2000": ice_cells1, "2000–2025": ice_cells2, "2025–2065": ice_cells3, "2065–2100": ice_cells4,}

# ------------------------------------------------------------
# Clean time for all periods
# ------------------------------------------------------------
ice_cells = {k: clean_time(v) for k, v in ice_cells.items()}

# ------------------------------------------------------------
# Rolling smoothing (5-day mean)
# ------------------------------------------------------------
ice_cells_smooth = {k: v.rolling(time = 5, center = True).mean() for k, v in ice_cells.items()}

# ============================================================
# DAILY TIME SERIES PLOTS
# ============================================================

# --- Plot 1: 1970–1975 ---
plt.figure(figsize = (10,5))
for year in range(1970, 1976):
    subset = ice_cells_smooth["1970–2000"].sel(
        time = slice(f"{year}-01-01", f"{year}-12-31")
    )
    if subset.size > 0:
        plt.plot(subset.time, subset.values, label = str(year), alpha = 0.7, color = "steelblue")

plt.title(f"Daily Sea Ice Coverage (1970–1975, 5-day mean)\nEns {ens_tag}")
plt.ylabel("Grid cells ≥30% sea ice")
plt.xlabel("Date")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: 2020–2025 ---
plt.figure(figsize = (10,5))
for year in range(2020, 2026):
    subset = ice_cells_smooth["2000–2025"].sel(time = slice(f"{year}-01-01", f"{year}-12-31"))
    if subset.size > 0:
        plt.plot(subset.time, subset.values, label = str(year), alpha = 0.7, color = "darkorange")

plt.title(f"Daily Sea Ice Coverage (2020–2025, 5-day mean)\nEns {ens_tag}")
plt.ylabel("Grid cells ≥30% sea ice")
plt.xlabel("Date")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

# --- Plot 3: 2095–2100 ---
plt.figure(figsize = (10,5))
for year in range(2095, 2101):
    subset = ice_cells_smooth["2065–2100"].sel(time = slice(f"{year}-01-01", f"{year}-12-31"))
    if subset.size > 0:
        plt.plot(subset.time, subset.values, label = str(year), alpha = 0.7, color = "firebrick")

plt.title(f"Daily Sea Ice Coverage (2095–2100, 5-day mean)\nEns {ens_tag}")
plt.ylabel("Grid cells ≥30% sea ice")
plt.xlabel("Date")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

# ============================================================
# MONTHLY CLIMATOLOGY
# ============================================================

def monthly_climatology(da, start_year, end_year):
    subset = da.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
    return subset.groupby("time.month").mean().compute()

# --- Compute climatologies ---
clim_1970s = monthly_climatology(ice_cells["1970–2000"], 1970, 1975)
clim_2020s = monthly_climatology(ice_cells["2000–2025"], 2020, 2025)
clim_2090s = monthly_climatology(ice_cells["2065–2100"], 2095, 2100)

# --- Smooth with 3-month rolling ---
clim_1970s_s = clim_1970s.rolling(month = 3, center = True).mean()
clim_2020s_s = clim_2020s.rolling(month = 3, center = True).mean()
clim_2090s_s = clim_2090s.rolling(month = 3, center = True).mean()

# --- Plot climatology ---
plt.figure(figsize = (9,5))
plt.plot(clim_1970s_s.month, clim_1970s_s, label = "1970–1975", color = "steelblue", linewidth = 2)
plt.plot(clim_2020s_s.month, clim_2020s_s, label = "2020–2025", color = "darkorange", linewidth = 2)
plt.plot(clim_2090s_s.month, clim_2090s_s, label = "2095–2100", color = "firebrick", linewidth = 2)

plt.title(f"Seasonal Cycle of Sea Ice Coverage\nEns {ens_tag}")
plt.xlabel("Month")
plt.ylabel("Grid cells ≥30% sea ice")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Done - Confirmed
# ============================================================
# Single-ensemble compressed daily sea ice view
# Ensemble: 1011.001
# ============================================================

ens_tag = "1011.001"

# ------------------------------------------------------------
# Helper: convert CFTime → datetime64 (robust for noleap)
# ------------------------------------------------------------
def clean_time(da):
    da = da.copy()
    if "cftime" in str(type(da.time.values[0])).lower():
        da["time"] = da.indexes["time"].to_datetimeindex()
    return da.compute()

# ------------------------------------------------------------
# Clean datasets
# ------------------------------------------------------------
ice_cells1 = clean_time(ice_cells1)
ice_cells2 = clean_time(ice_cells2)
ice_cells3 = clean_time(ice_cells3)
ice_cells4 = clean_time(ice_cells4)

# ------------------------------------------------------------
# Rolling smoothing (5-day mean)
# ------------------------------------------------------------
ice_cells1_smooth = ice_cells1.rolling(time = 5, center = True).mean()
ice_cells2_smooth = ice_cells2.rolling(time = 5, center = True).mean()
ice_cells3_smooth = ice_cells3.rolling(time = 5, center = True).mean()
ice_cells4_smooth = ice_cells4.rolling(time = 5, center = True).mean()

# ------------------------------------------------------------
# Helper: collapse each period to a continuous block
# ------------------------------------------------------------
def make_block(da, start_year, end_year):
    """
    Convert a time series slice into a uniform-length block.
    X-axis is artificial and only preserves relative timing.
    """
    subset = da.sel(time = slice(f"{start_year}-01-01", f"{end_year}-12-31"))
    n = subset.time.size
    x = np.linspace(0, 6, n)  # represent a ~6-year period
    return x, subset.values

# ------------------------------------------------------------
# Extract blocks (single ensemble)
# ------------------------------------------------------------
x1, y1 = make_block(ice_cells1_smooth, 1970, 1975)
x2, y2 = make_block(ice_cells2_smooth, 2020, 2025)
x3, y3 = make_block(ice_cells4_smooth, 2095, 2100)

# ------------------------------------------------------------
# Shift blocks along x-axis so they don’t overlap
# ------------------------------------------------------------
shift2 = x1[-1] + 1
shift3 = shift2 + x2[-1] + 1

x2 = x2 + shift2
x3 = x3 + shift3

# ------------------------------------------------------------
# Plot combined compressed timeline
# ------------------------------------------------------------
plt.figure(figsize = (12,5))

plt.plot(x1, y1, label = "1970–1975", color = "steelblue")
plt.plot(x2, y2, label = "2020–2025", color = "darkorange")
plt.plot(x3, y3, label = "2095–2100", color = "firebrick")

# Pretty x-axis labels at block midpoints
xticks = [x1.mean(), x2.mean(), x3.mean()]
xlabels = ["1970–1975", "2020–2025", "2095–2100"]
plt.xticks(xticks, xlabels)

plt.title("Daily Sea Ice Coverage (5-day mean, period-compressed view)\n" f"CESM2 Ensemble {ens_tag}")
plt.ylabel("Grid cells ≥30% sea ice")
plt.xlabel("Period")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Done - Confirmed
# ============================================================
# Single-ensemble overlaid 6-year sea ice coverage
# Ensemble: 1011.001
# ============================================================

ens_tag = "1011.001"

# ------------------------------------------------------------
# Helper: convert CFTime → datetime64
# ------------------------------------------------------------
def clean_time(da):
    da = da.copy()
    if not np.issubdtype(da.time.dtype, np.datetime64):
        da["time"] = xr.decode_cf(da.to_dataset(name = "tmp")).time
    return da.compute()

# ------------------------------------------------------------
# Clean datasets
# ------------------------------------------------------------
ice_cells1 = clean_time(ice_cells1)
ice_cells2 = clean_time(ice_cells2)
ice_cells3 = clean_time(ice_cells3)
ice_cells4 = clean_time(ice_cells4)

# ------------------------------------------------------------
# Rolling smoothing (5-day mean)
# ------------------------------------------------------------
def smooth(da):
    return da.rolling(time = 5, center = True).mean()

ice_cells1_smooth = smooth(ice_cells1)
ice_cells2_smooth = smooth(ice_cells2)
ice_cells3_smooth = smooth(ice_cells3)
ice_cells4_smooth = smooth(ice_cells4)

# ------------------------------------------------------------
# Extract 6-year periods
# ------------------------------------------------------------
def extract_period(da, start, end):
    return da.sel(time = slice(f"{start}-01-01", f"{end}-12-31")).dropna(dim = "time")

p1 = extract_period(ice_cells1_smooth, 1970, 1975)
p2 = extract_period(ice_cells2_smooth, 2020, 2025)
p3 = extract_period(ice_cells4_smooth, 2095, 2100)

# ------------------------------------------------------------
# Normalize time axis (days since period start)
# ------------------------------------------------------------
def normalize_time(da):
    t_days = (da.time - da.time[0]) / np.timedelta64(1, "D")
    return xr.DataArray(da.values, dims = ["time"], coords = {"time": t_days})

p1n = normalize_time(p1)
p2n = normalize_time(p2)
p3n = normalize_time(p3)

# ------------------------------------------------------------
# Plot: overlaid periods
# ------------------------------------------------------------
plt.figure(figsize = (10,5))

plt.plot(p1n.time, p1n, label = "1970–1975", color = "steelblue", linewidth = 2)
plt.plot(p2n.time, p2n, label = "2020–2025", color = "darkorange", linewidth = 2)
plt.plot(p3n.time, p3n, label = "2095–2100", color = "firebrick", linewidth = 2)

plt.title("Sea Ice Coverage (≥30%) — Overlaid 6-Year Periods\n" f"CESM2 Ensemble {ens_tag}", fontsize = 14)
plt.xlabel("Days since period start (~6 years)")
plt.ylabel("Grid cells ≥30% sea ice")
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

## Snow Depth 1970-2100

In [ ]:
# Done - Confirmed
def run_snow_pipeline(ens_tag, basedir):
    """
    Run full snow analysis & plotting for one ensemble member.
    ens_tag e.g. "1011.001"
    """

    VERIFY_PLOTS = True   # set False for batch runs

    # Optional: FDR correction
    try:
        from statsmodels.stats.multitest import fdrcorrection
        HAS_FDR = True
    except Exception:
        HAS_FDR = False
        print("statsmodels.fdrcorrection not found — continuing without FDR correction")

    print(f"\n=== Running SNOW ensemble {ens_tag} ===")

    # ------------------------
    # ICE FILES (for masking)
    # ------------------------
    ice_me = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_1970-2000_{ens_tag}.nc"
    ice_mr = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2000-2025_{ens_tag}.nc"
    ice_fe = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2025-2065_{ens_tag}.nc"
    ice_fl = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2065-2100_{ens_tag}.nc"

    ds_i1 = xr.open_dataset(ice_me, chunks = {"time":30})
    ds_i2 = xr.open_dataset(ice_mr, chunks = {"time":30})
    ds_i3 = xr.open_dataset(ice_fe, chunks = {"time":30})
    ds_i4 = xr.open_dataset(ice_fl, chunks = {"time":30})

    ice1 = ds_i1["aice_d"]
    ice2 = ds_i2["aice_d"]
    ice3 = ds_i3["aice_d"]
    ice4 = ds_i4["aice_d"]

    # ------------------------
    # FILES
    # ------------------------
    snow_me = f"{basedir}/snow_1970-2000_{ens_tag}.nc"
    snow_mr = f"{basedir}/snow_2000-2025_{ens_tag}.nc"
    snow_fe = f"{basedir}/snow_2025-2065_{ens_tag}.nc"
    snow_fl = f"{basedir}/snow_2065-2100_{ens_tag}.nc"

    var = "vsnon_d"        # snow depth (m)
    event_thresh = 0.10   # ≥ 0.1 m snow

    # ------------------------
    # LOAD DATA
    # ------------------------
    ds1 = xr.open_dataset(snow_me, chunks = {"time": 30})
    ds2 = xr.open_dataset(snow_mr, chunks = {"time": 30})
    ds3 = xr.open_dataset(snow_fe, chunks = {"time": 30})
    ds4 = xr.open_dataset(snow_fl, chunks = {"time": 30})

    snow1 = ds1[var]
    snow2 = ds2[var]
    snow3 = ds3[var]
    snow4 = ds4[var]

    # 🔹 REMOVE DUPLICATE TIMES (REQUIRED FOR CESM)
    ice1  = ice1.isel(time = ~ice1.get_index("time").duplicated())
    snow1 = snow1.isel(time = ~snow1.get_index("time").duplicated())
    
    ice2  = ice2.isel(time = ~ice2.get_index("time").duplicated())
    snow2 = snow2.isel(time = ~snow2.get_index("time").duplicated())
    
    ice3  = ice3.isel(time = ~ice3.get_index("time").duplicated())
    snow3 = snow3.isel(time = ~snow3.get_index("time").duplicated())
    
    ice4  = ice4.isel(time = ~ice4.get_index("time").duplicated())
    snow4 = snow4.isel(time = ~snow4.get_index("time").duplicated())
    
    ice1, snow1 = xr.align(ice1, snow1, join = "inner")
    ice2, snow2 = xr.align(ice2, snow2, join = "inner")
    ice3, snow3 = xr.align(ice3, snow3, join = "inner")
    ice4, snow4 = xr.align(ice4, snow4, join = "inner")

    ice_mask1 = ice1 >= 0.15
    ice_mask2 = ice2 >= 0.15
    ice_mask3 = ice3 >= 0.15
    ice_mask4 = ice4 >= 0.15

    snow1 = snow1.where(ice_mask1)
    snow2 = snow2.where(ice_mask2)
    snow3 = snow3.where(ice_mask3)
    snow4 = snow4.where(ice_mask4)

    # ---- lon/lat detection ----
    if ("lon" in snow1.coords) and ("lat" in snow1.coords):
        lon = snow1["lon"]
        lat = snow1["lat"]
    else:
        spatial = [d for d in snow1.dims if d != "time"]
        lat = snow1[spatial[0]]
        lon = snow1[spatial[1]]

    # ------------------------
    # PARAMETERS
    # ------------------------
    alpha = 0.05
    min_years = 8
    edge_low = 0.15
    edge_high = 0.85
    stipple_density = 0.12
    stipple_jitter = 0.25
    stipple_seed = 42

    # ------------------------
    # Annual fractions
    # ------------------------
    def annual_fraction(da, threshold):
        return (da >= threshold).resample(time = "YE").mean("time")

    ann1 = annual_fraction(snow1, event_thresh).compute()
    ann2 = annual_fraction(snow2, event_thresh).compute()
    ann3 = annual_fraction(snow3, event_thresh).compute()
    ann4 = annual_fraction(snow4, event_thresh).compute()

    # ------------------------
    # Mean fractions
    # ------------------------
    def frac_days(da, thresh):
        valid = da.notnull()
        events = (da >= thresh)
        return events.sum("time") / valid.sum("time")
    
    snow_frac1 = frac_days(snow1, event_thresh)
    snow_frac2 = frac_days(snow2, event_thresh)
    snow_frac3 = frac_days(snow3, event_thresh)
    snow_frac4 = frac_days(snow4, event_thresh)

    diff_2000v1970 = snow_frac2 - snow_frac1
    diff_2065v2000 = snow_frac3 - snow_frac2
    diff_2100v2065 = snow_frac4 - snow_frac3
    diff_2100v1970 = snow_frac4 - snow_frac1

    # detect spatial dims
    spatial_dims = [d for d in snow_frac1.dims if d != "time"]
    dim_y, dim_x = spatial_dims

    # ------------------------
    # Significance mask
    # ------------------------
    def significance_mask(annA, annB, fracA, fracB, alpha = 0.05, min_years = 8, use_fdr = HAS_FDR):

        countsA = np.sum(~np.isnan(annA.values), axis = 0)
        countsB = np.sum(~np.isnan(annB.values), axis = 0)
        sufficient = (countsA >= min_years) & (countsB >= min_years)

        A = annA.values
        B = annB.values
        
        countsA = np.sum(~np.isnan(A), axis = 0)
        countsB = np.sum(~np.isnan(B), axis = 0)
        
        valid_cells = (countsA >= min_years) & (countsB >= min_years)
        
        pval = np.full(A.shape[1:], np.nan)
        
        if valid_cells.any():
            tstat_tmp, pval_tmp = ttest_ind(A[:, valid_cells], B[:, valid_cells], axis = 0, equal_var = False, nan_policy = "omit")
            pval[valid_cells] = pval_tmp

        pval_da = xr.DataArray(pval, coords = {dim_y: fracA[dim_y], dim_x: fracA[dim_x]}, 
                               dims = (dim_y, dim_x)).where(sufficient, other = 1.0)

        if use_fdr:
            flatp = pval_da.values.ravel()
            valid = np.isfinite(flatp)
            reject = np.zeros_like(flatp, dtype = bool)
            if valid.any():
                reject_valid, _ = fdrcorrection(flatp[valid], alpha = alpha)
                reject[valid] = reject_valid
            stat_sig = xr.DataArray(reject.reshape(pval_da.shape), coords = pval_da.coords, dims = pval_da.dims)
        else:
            stat_sig = pval_da < alpha

        sigmaA = annA.std("time")
        decline = fracA - fracB
        mag_mask = decline >= sigmaA
        decline_dir = fracB < fracA
        edge_mask = (fracA > edge_low) & (fracA < edge_high)

        final = stat_sig & mag_mask & decline_dir & edge_mask
        return final.fillna(False).astype(bool)

    sig_2000v1970 = significance_mask(ann1, ann2, snow_frac1, snow_frac2)
    sig_2065v2000 = significance_mask(ann2, ann3, snow_frac2, snow_frac3)
    sig_2100v2065 = significance_mask(ann3, ann4, snow_frac3, snow_frac4)

    # ------------------------
    # Stippling helper
    # ------------------------
    def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):

        np.random.seed(seed)
        ys, xs = np.where(mask_da.values)
        if len(xs) == 0:
            return

        if lon_da.ndim == 2:
            sig_lon = lon_da.values[ys, xs]
            sig_lat = lat_da.values[ys, xs]
        else:
            sig_lon = lon_da.values[xs]
            sig_lat = lat_da.values[ys]

        keep = np.random.rand(len(sig_lon)) < density
        if keep.sum() == 0:
            return

        ax.scatter(sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum()), 
                   sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum()), 
                   s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 6)

    # ------------------------
    # Plotting
    # ------------------------
    proj = ccrs.NorthPolarStereo()
    transform = ccrs.PlateCarree()
    extent = [-180, 180, 50, 90]

    def setup_map(ax):
        ax.set_extent(extent, crs = transform)
        ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3)
        ax.coastlines(linewidth = 0.9)
        ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)

    # ---- FIGURE 1: mean fractions ----
    fig1, axs = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

    maps = [snow_frac1, snow_frac2, snow_frac3, snow_frac4]
    titles = [f"Fraction ≥0.1 m Snow (1970–2000)\nEns {ens_tag}", "Fraction ≥0.1 m Snow (2000–2025)",
              "Fraction ≥0.1 m Snow (2025–2065)", "Fraction ≥0.1 m Snow (2065–2100)"]
    masks = [None, sig_2000v1970, sig_2065v2000, sig_2100v2065]

    for ax, data, title, mask in zip(axs.flat, maps, titles, masks):
        setup_map(ax)
        p = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "cividis", vmin = 0, vmax = 1, shading = "auto")
        ax.set_title(title)
        if mask is not None:
            add_stipple(ax, lon, lat, mask, density = stipple_density, jitter = stipple_jitter, seed = stipple_seed)

    cbar = fig1.colorbar(p, ax = axs.ravel().tolist(), shrink = 0.78, pad = 0.02)
    cbar.set_label("Fraction of Days ≥0.1 m Snow")

    fig1.savefig(f"cesm2_snow_fraction_{ens_tag}.png", dpi = 300, bbox_inches = "tight")
    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig1)

    # ------------------------
    # FIGURE 2 — Difference Maps
    # ------------------------
    diffs = [diff_2000v1970, diff_2100v2065, diff_2100v1970]

    titles_diff = ["Change (2000–2025 minus 1970–2000)", "Change (2065–2100 minus 2025–2065)", "Change (2065–2100 minus 1970–2000)"]

    vmin = min(float(d.min()) for d in diffs)
    vmax = max(float(d.max()) for d in diffs)
    lim = max(abs(vmin), abs(vmax))
    norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

    fig2, axs2 = plt.subplots(ncols = 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

    for ax, data, title in zip(axs2, diffs, titles_diff):
        setup_map(ax)
        p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", shrink = 0.85, pad = 0.08)
    cbar2.set_label("Change in Fraction of Days with ≥0.1 m Snow", fontsize = 12, labelpad = 6)

    fig2.savefig(f"cesm2_snow_diff_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()

    plt.close(fig2)

    # ------------------------
    # Cleanup
    # ------------------------
    for ds in [ds1, ds2, ds3, ds4]:
        ds.close()

    for ds in [ds_i1, ds_i2, ds_i3, ds_i4]:
        ds.close()

basedir = "/glade/work/xliu1/cesm2_sosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

for ens in ensemble_members:
    run_snow_pipeline(ens, basedir)

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# -----------------------------
# SETTINGS
# -----------------------------
basedir = "/glade/work/xliu1/cesm2_sosi_regrid"
ensemble_members = ["1011.001","1031.002","1051.003","1071.004","1091.005",
                    "1111.006","1131.007","1151.008","1171.009","1191.010"]

snr_threshold = 1.0
snow_thresh = 0.10
ice_thresh  = 0.15

# -----------------------------
# LOAD FUNCTION
# -----------------------------
def annual_snow_fraction(snow_file, ice_file):

    ds_snow = xr.open_dataset(snow_file, chunks = {"time":30})
    ds_ice  = xr.open_dataset(ice_file,  chunks = {"time":30})

    snow = ds_snow["vsnon_d"]
    ice  = ds_ice["aice_d"]

    lon = ds_snow["lon"]
    lat = ds_snow["lat"]

    snow = snow.isel(time = ~snow.get_index("time").duplicated())
    ice  = ice.isel(time = ~ice.get_index("time").duplicated())

    snow, ice = xr.align(snow, ice, join = "inner")

    # mask snow to sea ice
    snow = snow.where(ice >= ice_thresh)

    # snow event boolean
    snow_event = snow >= snow_thresh

    # ⭐ KEY: keep yearly variability
    yearly = snow_event.groupby("time.year").mean("time")

    # only keep grid cells ice-covered in that year
    ice_year = (ice >= ice_thresh).groupby("time.year").mean("time")

    return yearly.where(ice_year > 0.5), lon, lat

# -----------------------------
# LOAD ALL ENSEMBLES
# -----------------------------
frac1_all, frac2_all, frac3_all, frac4_all = [], [], [], []

for ens in ensemble_members:

    print(f"Loading ensemble {ens}")

    snow1 = f"{basedir}/snow_1970-2000_{ens}.nc"
    snow2 = f"{basedir}/snow_2000-2025_{ens}.nc"
    snow3 = f"{basedir}/snow_2025-2065_{ens}.nc"
    snow4 = f"{basedir}/snow_2065-2100_{ens}.nc"

    ice1 = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_1970-2000_{ens}.nc"
    ice2 = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2000-2025_{ens}.nc"
    ice3 = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2025-2065_{ens}.nc"
    ice4 = f"/glade/work/xliu1/cesm2_seaice_regrid/aice_2065-2100_{ens}.nc"

    f1,lon,lat = annual_snow_fraction(snow1, ice1)
    f2,_,_ = annual_snow_fraction(snow2, ice2)
    f3,_,_ = annual_snow_fraction(snow3, ice3)
    f4,_,_ = annual_snow_fraction(snow4, ice4)

    frac1_all.append(f1)
    frac2_all.append(f2)
    frac3_all.append(f3)
    frac4_all.append(f4)

frac1_all = xr.concat(frac1_all, dim = "ens").assign_coords(ens = ensemble_members)
frac2_all = xr.concat(frac2_all, dim = "ens").assign_coords(ens = ensemble_members)
frac3_all = xr.concat(frac3_all, dim = "ens").assign_coords(ens = ensemble_members)
frac4_all = xr.concat(frac4_all, dim = "ens").assign_coords(ens = ensemble_members)

# =====================================================
# PERIOD MEANS PER ENSEMBLE MEMBER
# =====================================================
mean1_ens = frac1_all.mean("year")
mean2_ens = frac2_all.mean("year")
mean3_ens = frac3_all.mean("year")
mean4_ens = frac4_all.mean("year")

# =====================================================
# SIGNAL / NOISE / SNR 
# =====================================================
signal1 = mean2_ens.mean("ens") - mean1_ens.mean("ens")
signal2 = mean4_ens.mean("ens") - mean3_ens.mean("ens")
signal3 = mean4_ens.mean("ens") - mean1_ens.mean("ens")

noise1 = (mean2_ens - mean1_ens).std("ens")
noise2 = (mean4_ens - mean3_ens).std("ens")
noise3 = (mean4_ens - mean1_ens).std("ens")

eps = 1e-6
noise1_safe = noise1.where(noise1 > eps)
noise2_safe = noise2.where(noise2 > eps)
noise3_safe = noise3.where(noise3 > eps)

snr1 = signal1 / noise1_safe
snr2 = signal2 / noise2_safe
snr3 = signal3 / noise3_safe

mask1 = np.abs(snr1) > snr_threshold
mask2 = np.abs(snr2) > snr_threshold
mask3 = np.abs(snr3) > snr_threshold

print("\nSNR1:", float(snr1.min().compute()), float(snr1.max().compute()))
print("SNR2:", float(snr2.min().compute()), float(snr2.max().compute()))
print("SNR3:", float(snr3.min().compute()), float(snr3.max().compute()))

print("\nNoise1:", float(noise1.min().compute()), float(noise1.max().compute()))
print("Noise2:", float(noise2.min().compute()), float(noise2.max().compute()))
print("Noise3:", float(noise3.min().compute()), float(noise3.max().compute()))

print("\nSignal1:", float(signal1.min().compute()), float(signal1.max().compute()))
print("Signal2:", float(signal2.min().compute()), float(signal2.max().compute()))
print("Signal3:", float(signal3.min().compute()), float(signal3.max().compute()))

# =====================================================
# FIGURE 1 — ENSEMBLE MEAN STATE
# =====================================================
def collapse_state(frac):
    return frac.mean(dim = [d for d in frac.dims if d in ["ens", "year"]])

mean1 = collapse_state(frac1_all)
mean2 = collapse_state(frac2_all)
mean3 = collapse_state(frac3_all)
mean4 = collapse_state(frac4_all)

proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

fig1, axs1 = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

state_maps = [mean1, mean2, mean3, mean4]
state_titles = ["1970–2000 Ensemble Mean Snow", "2000–2025 Ensemble Mean Snow",
                "2025–2065 Ensemble Mean Snow", "2065–2100 Ensemble Mean Snow"]

for ax, data, title in zip(axs1.flat, state_maps, state_titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p1 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "cividis", vmin = 0, vmax = 1, shading = "auto", zorder = 1)
    ax.set_title(title)

cbar1 = fig1.colorbar(p1, ax = axs1.ravel().tolist(), pad = 0.02)
cbar1.set_label("Fraction of Days with Snow Depth ≥0.1 m")
fig1.savefig("cesm2_snow_mean_state.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 2 — ENSEMBLE MEAN CHANGE + SNR STIPPLING
# =====================================================

# -----------------------------
# HISTOGRAM OF TOTAL CHANGE
# -----------------------------
sig = signal3

baseline_ice = (mean1_ens.mean("ens")) > 0.15
vals = sig.where(baseline_ice).data.compute().ravel()
vals = vals[np.isfinite(vals)]

plt.figure(figsize = (8, 5))

plt.hist(vals, bins = 80)
plt.axvline(0, linestyle = "--")
plt.title("Snow change distribution — sea-ice regions")
plt.xlabel("Δ Fraction of Days with Snow Depth ≥0.1 m")
plt.ylabel("Count")

plt.savefig("cesm2_snow_histogram.png", dpi = 300, bbox_inches = "tight")
plt.show()

def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):
    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)
    if len(xs) == 0:
        return
    if getattr(lon_da, "ndim", None) == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 30)

fig2, axs2 = plt.subplots(1, 3, figsize = (18,6), subplot_kw = dict(projection = proj), constrained_layout = True)

diff_maps = [signal1, signal2, signal3]
diff_masks = [mask1, mask2, mask3]
diff_titles = ["2000–2025 minus 1970–2000", "2065–2100 minus 2025–2065", "2065–2100 minus 1970–2000"]

vmin = min(float(m.min().compute()) for m in diff_maps)
vmax = max(float(m.max().compute()) for m in diff_maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, mask, title in zip(axs2, diff_maps, diff_masks, diff_titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto", zorder = 1)
    add_stipple(ax, lon, lat, mask, density = 0.05, size = 4, alpha = 0.7)
    ax.set_title(title)

cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.07)
cbar2.set_label("Change in Fraction of Days with Snow Depth ≥0.1 m")
fig2.savefig("cesm2_snow_mean_change.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 3 — SIGNAL MAPS
# =====================================================
fig3, axs3 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

signal_maps = [signal1, signal2, signal3]
titles_signal = ["Signal: 2000–2025 − 1970–2000", "Signal: 2065–2100 − 2025–2065", "Signal: 2065–2100 − 1970–2000"]

vmin = min(float(m.min().compute()) for m in signal_maps)
vmax = max(float(m.max().compute()) for m in signal_maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, title in zip(axs3, signal_maps, titles_signal):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3)
    ax.coastlines(linewidth = 0.9)

    p3 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")
    ax.set_title(title)

cbar3 = fig3.colorbar(p3, ax = axs3, orientation = "horizontal", pad = 0.07)
cbar3.set_label("Signal (Ensemble Mean Snow Change)")
fig3.savefig("cesm2_snow_signal.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 4 — NOISE MAPS
# =====================================================
fig4, axs4 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

noise_maps = [noise1, noise2, noise3]
titles_noise = ["Noise: 2000–2025 − 1970–2000", "Noise: 2065–2100 − 2025–2065", "Noise: 2065–2100 − 1970–2000"]

all_noise = np.concatenate([m.data.compute().ravel() for m in noise_maps])
all_noise = all_noise[np.isfinite(all_noise)]

vmax_noise = np.percentile(all_noise, 99)   # or 98 if still dark

for ax, data, title in zip(axs4, noise_maps, titles_noise):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3)
    ax.coastlines(linewidth = 0.9)

    p4 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "magma", vmin = 0, vmax = vmax_noise, shading = "auto")
    ax.set_title(title)

cbar4 = fig4.colorbar(p4, ax = axs4, orientation = "horizontal", pad = 0.07)
cbar4.set_label("Internal Variability (σ across ensemble)")
fig4.savefig("cesm2_snow_noise.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 5 — SNR MAPS
# =====================================================
fig5, axs5 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

snr_maps = [snr1, snr2, snr3]
titles_snr = ["SNR: 2000–2025 − 1970–2000", "SNR: 2065–2100 − 2025–2065", "SNR: 2065–2100 − 1970–2000"]

# symmetric limits around zero
snr_lim = max(abs(float(m.min().compute())) for m in snr_maps)
snr_lim = max(snr_lim, max(abs(float(m.max().compute())) for m in snr_maps))
snr_lim = min(snr_lim, 5)

norm = TwoSlopeNorm(vcenter = 0, vmin = -snr_lim, vmax = snr_lim)

for ax, data, title in zip(axs5, snr_maps, titles_snr):

    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3)
    ax.coastlines(linewidth = 0.9)

    p5 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")

    ax.set_title(title)

cbar5 = fig5.colorbar(p5, ax = axs5, orientation = "horizontal",pad = 0.07)
cbar5.set_label("Signal-to-Noise Ratio")

fig5.savefig("cesm2_snow_snr.png", dpi = 300, bbox_inches = "tight")
plt.show()

## Precipitation 1970-2100

In [ ]:
# Done - Confirmed
import warnings

warnings.filterwarnings("ignore", 
                        message = "After omitting NaNs, one or more axis-slices of one or more sample arguments is too small*")

def run_rain_pipeline(ens_tag, basedir, VERIFY_PLOTS = True):
    """
    Run full rain analysis & plotting for one ensemble member.
    ens_tag e.g. "1011.001"
    """

    print(f"\n=== Running RAIN ensemble {ens_tag} ===")

    # =========================
    # FILE PATHS
    # =========================
    rain_me = f"{basedir}/rain_1970-2000_{ens_tag}.nc"
    rain_mr = f"{basedir}/rain_2000-2025_{ens_tag}.nc"
    rain_fe = f"{basedir}/rain_2025-2065_{ens_tag}.nc"
    rain_fl = f"{basedir}/rain_2065-2100_{ens_tag}.nc"

    var = "rain_d"
    rain_thresh = 0.10   # ≥ 1 mm/day

    # =========================
    # LOAD DATA
    # =========================
    r1 = xr.open_dataset(rain_me, chunks = {"time": 30})
    r2 = xr.open_dataset(rain_mr, chunks = {"time": 30})
    r3 = xr.open_dataset(rain_fe, chunks = {"time": 30})
    r4 = xr.open_dataset(rain_fl, chunks = {"time": 30})

    # =========================
    # FRACTION OF RAIN DAYS
    # =========================
    rain_frac1 = (r1[var] >= rain_thresh).mean("time")
    rain_frac2 = (r2[var] >= rain_thresh).mean("time")
    rain_frac3 = (r3[var] >= rain_thresh).mean("time")
    rain_frac4 = (r4[var] >= rain_thresh).mean("time")

    lon = rain_frac1.lon
    lat = rain_frac1.lat

    # =========================
    # ANNUAL MEANS (for t-test)
    # =========================
    ann1 = r1[var].resample(time = "YE").mean()
    ann2 = r2[var].resample(time = "YE").mean()
    ann3 = r3[var].resample(time = "YE").mean()
    ann4 = r4[var].resample(time = "YE").mean()

    # =========================
    # DIFFERENCES
    # =========================
    d_2000v1970 = rain_frac2 - rain_frac1
    d_2065v2000 = rain_frac3 - rain_frac2
    d_2100v2065 = rain_frac4 - rain_frac3
    d_2100v1970 = rain_frac4 - rain_frac1

    # =========================
    # SIGNIFICANCE FUNCTION
    # =========================
    def significance_mask(annA, annB, fracA, fracB, alpha = 0.10, min_years = 10, edge_low = 0.10, edge_high = 0.90):

        countsA = np.sum(~np.isnan(annA.values), axis = 0)
        countsB = np.sum(~np.isnan(annB.values), axis = 0)
        sufficient = (countsA >= min_years) & (countsB >= min_years)

        t, p = ttest_ind(annA.values, annB.values, axis = 0, equal_var = False, nan_policy = "omit")

        pval_da = xr.DataArray(p, coords = fracA.coords, dims = fracA.dims).where(sufficient, other = 1.0)

        stat_sig = pval_da < alpha

        diff = fracB - fracA
        mag_mask = diff >= 0.10          # ≥10% increase
        increase_dir = fracB > fracA
        edge_mask = (fracA > edge_low) & (fracA < edge_high)

        final = stat_sig & mag_mask & increase_dir & edge_mask
        return final.fillna(False).astype(bool)

    sig_2000v1970 = significance_mask(ann1, ann2, rain_frac1, rain_frac2)
    sig_2065v2000 = significance_mask(ann2, ann3, rain_frac2, rain_frac3)
    sig_2100v2065 = significance_mask(ann3, ann4, rain_frac3, rain_frac4)

    # =========================
    # MAP SETUP
    # =========================
    proj = ccrs.NorthPolarStereo()
    transform = ccrs.PlateCarree()
    extent = [-180, 180, 50, 90]

    def setup_map(ax):
        ax.set_extent(extent, crs = transform)
        ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
        ax.coastlines(linewidth = 0.9, zorder = 6)
        ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)
        return ax

    # =========================
    # STIPPLING
    # =========================
    def add_stipple(ax, lon, lat, mask, density = 0.10, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):

        np.random.seed(seed)
        ys, xs = np.where(mask.values)
        if len(xs) == 0:
            return

        sig_lon = lon.values[xs]
        sig_lat = lat.values[ys]

        keep = np.random.rand(len(sig_lon)) < density
        sig_lon = sig_lon[keep]
        sig_lat = sig_lat[keep]

        sig_lon += np.random.uniform(-jitter, jitter, len(sig_lon))
        sig_lat += np.random.uniform(-jitter, jitter, len(sig_lat))

        ax.scatter(sig_lon, sig_lat, s = size, color = color, alpha = alpha, transform = transform, marker = ".", zorder = 6)

    # =========================
    # FIGURE 1 — MEAN FRACTIONS
    # =========================
    fig1, axs = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

    maps = [rain_frac1, rain_frac2, rain_frac3, rain_frac4]
    titles = [f"Fraction ≥1 mm/day Rain (1970–2000)\nEns {ens_tag}", "Fraction ≥1 mm/day Rain (2000–2025)",
              "Fraction ≥1 mm/day Rain (2025–2065)", "Fraction ≥1 mm/day Rain (2065–2100)"]
    masks = [None, sig_2000v1970, sig_2065v2000, sig_2100v2065]

    for ax, data, title, mask in zip(axs.flat, maps, titles, masks):
        setup_map(ax)
        p = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "cividis", vmin = 0, vmax = 1, shading = "auto")
        ax.set_title(title, fontsize = 13)

        if mask is not None:
            add_stipple(ax, lon, lat, mask)

    cbar = fig1.colorbar(p, ax = axs.ravel().tolist(), shrink = 0.78, pad = 0.02)
    cbar.set_label("Fraction of Days with ≥1 mm/day Rain")

    fig1.savefig(f"cesm2_rain_fraction_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig1)

    # =========================
    # FIGURE 2 — DIFFERENCES
    # =========================
    diffs = [d_2000v1970, d_2065v2000, d_2100v1970]
    titles_diff = ["Change (2000–2025 minus 1970–2000)", "Change (2065–2100 minus 2025–2065)", "Change (2065–2100 minus 1970–2000)"]

    vmin = min(float(d.min()) for d in diffs)
    vmax = max(float(d.max()) for d in diffs)
    lim = max(abs(vmin), abs(vmax))
    norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

    fig2, axs2 = plt.subplots(ncols = 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

    for ax, data, title in zip(axs2, diffs, titles_diff):
        setup_map(ax)
        p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", shrink = 0.85, pad = 0.08)
    cbar2.set_label("Change in Fraction of Rain Days")

    fig2.savefig(f"cesm2_rain_diff_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig2)

    # =========================
    # CLEANUP
    # =========================
    for ds in [r1, r2, r3, r4]:
        ds.close()

basedir = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005", 
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

for ens in ensemble_members:
    run_rain_pipeline(ens, basedir, VERIFY_PLOTS = True)

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# -----------------------------
# SETTINGS
# -----------------------------
basedir = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

rain_threshold = 0.1
snr_threshold = 1.0

# -----------------------------
# LOAD FUNCTION (MATCH ICE)
# -----------------------------
def annual_fraction(file):

    ds = xr.open_dataset(file, chunks = {"time":30})
    rain = ds["rain_d"]

    event = rain >= rain_threshold

    yearly = event.groupby("time.year").mean("time")

    return yearly

# -----------------------------
# LOAD ENSEMBLES
# -----------------------------
frac1_all, frac2_all, frac3_all, frac4_all = [], [], [], []

for ens in ensemble_members:
    print(f"Loading ensemble {ens}")
    frac1_all.append(annual_fraction(f"{basedir}/rain_1970-2000_{ens}.nc"))
    frac2_all.append(annual_fraction(f"{basedir}/rain_2000-2025_{ens}.nc"))
    frac3_all.append(annual_fraction(f"{basedir}/rain_2025-2065_{ens}.nc"))
    frac4_all.append(annual_fraction(f"{basedir}/rain_2065-2100_{ens}.nc"))

frac1_all = xr.concat(frac1_all, dim = "ens").assign_coords(ens = ensemble_members)
frac2_all = xr.concat(frac2_all, dim = "ens").assign_coords(ens = ensemble_members)
frac3_all = xr.concat(frac3_all, dim = "ens").assign_coords(ens = ensemble_members)
frac4_all = xr.concat(frac4_all, dim = "ens").assign_coords(ens = ensemble_members)

lon = frac1_all.lon
lat = frac1_all.lat

# =====================================================
# FIGURE 1 — ENSEMBLE MEAN STATE  (IDENTICAL TO ICE)
# =====================================================
def collapse_state(frac):
    return frac.mean(dim = [d for d in frac.dims if d in ["ens", "year"]])

mean1 = collapse_state(frac1_all)
mean2 = collapse_state(frac2_all)
mean3 = collapse_state(frac3_all)
mean4 = collapse_state(frac4_all)

proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

fig1, axs1 = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

maps = [mean1, mean2, mean3, mean4]
titles = ["1970–2000 Ensemble Mean", "2000–2025 Ensemble Mean", "2025–2065 Ensemble Mean", "2065–2100 Ensemble Mean"]

for ax, data, title in zip(axs1.flat, maps, titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25) 

    p1 = ax.pcolormesh(data.lon,data.lat,data, transform = transform, cmap = "cividis", 
                       vmin = 0, vmax = 1, shading = "nearest", zorder = 1)

    ax.set_title(title)

cbar1 = fig1.colorbar(p1, ax = axs1.ravel().tolist(), pad = 0.02)
cbar1.set_label("Fraction of Days with Rain Event")
fig1.savefig("cesm2_rain_mean_state.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 2 — SIGNAL / NOISE / SNR (MATCH ICE EXACTLY)
# =====================================================

# ---- PERIOD MEANS PER ENSEMBLE ----
mean1_ens = frac1_all.mean("year")
mean2_ens = frac2_all.mean("year")
mean3_ens = frac3_all.mean("year")
mean4_ens = frac4_all.mean("year")

# ---- SIGNAL ----
signal1 = mean2_ens.mean("ens") - mean1_ens.mean("ens")
signal2 = mean4_ens.mean("ens") - mean3_ens.mean("ens")
signal3 = mean4_ens.mean("ens") - mean1_ens.mean("ens")

# ---- NOISE ----
noise1 = (mean2_ens - mean1_ens).std("ens")
noise2 = (mean4_ens - mean3_ens).std("ens")
noise3 = (mean4_ens - mean1_ens).std("ens")

# ---- SAFE SNR ----
eps = 1e-6
snr1 = signal1 / noise1.where(noise1 > eps)
snr2 = signal2 / noise2.where(noise2 > eps)
snr3 = signal3 / noise3.where(noise3 > eps)

mask1 = abs(snr1) > snr_threshold
mask2 = abs(snr2) > snr_threshold
mask3 = abs(snr3) > snr_threshold

print("\nSNR1 range:", float(snr1.min().compute()), float(snr1.max().compute()))
print("SNR2 range:", float(snr2.min().compute()), float(snr2.max().compute()))
print("SNR3 range:", float(snr3.min().compute()), float(snr3.max().compute()))

print("\nNoise1 min/max:", float(noise1.min().compute()), float(noise1.max().compute()))
print("Noise2 min/max:", float(noise2.min().compute()), float(noise2.max().compute()))
print("Noise3 min/max:", float(noise3.min().compute()), float(noise3.max().compute()))

print("\nSignal1 min/max:", float(signal1.min().compute()), float(signal1.max().compute()))
print("Signal2 min/max:", float(signal2.min().compute()), float(signal2.max().compute()))
print("Signal3 min/max:", float(signal3.min().compute()), float(signal3.max().compute()))

# =====================================================
# HISTOGRAMS (IDENTICAL TO ICE)
# =====================================================
sig = signal3

vals_all = sig.data.compute().ravel()
vals_all = vals_all[np.isfinite(vals_all)]

baseline = mean1_ens.mean("ens")
rain_mask = baseline > 0.05

vals_event = sig.where(rain_mask).data.compute().ravel()
vals_event = vals_event[np.isfinite(vals_event)]

fig,axs = plt.subplots(2, 1, figsize = (8, 8), constrained_layout = True)

axs[0].hist(vals_all,bins = 80)
axs[0].axvline(0,linestyle = "--")
axs[0].set_title("Change distribution — ALL grid cells")

axs[1].hist(vals_event,bins = 80)
axs[1].axvline(0,linestyle = "--")
axs[1].set_title("Change distribution — rain cells only")

plt.savefig("cesm2_rain_histograms.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# STIPPLE FUNCTION (IDENTICAL TO ICE)
# =====================================================
def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):

    np.random.seed(seed)
    ys,xs = np.where(mask_da.values)
    if len(xs) == 0:
        return

    if lon_da.ndim == 2:
        sig_lon = lon_da.values[ys,xs]
        sig_lat = lat_da.values[ys,xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter ,keep.sum())

    ax.scatter(sig_lon,sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 30)

# =====================================================
# FIGURE 2 — CHANGE MAPS WITH STIPPLING
# =====================================================
fig2, axs2 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

maps = [signal1, signal2, signal3]
masks = [mask1, mask2, mask3]
titles = ["2000–2025 minus 1970–2000", "2065–2100 minus 2025–2065", "2065–2100 minus 1970–2000"]

vmin = min(float(m.min().compute()) for m in maps)
vmax = max(float(m.max().compute()) for m in maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, mask, title in zip(axs2, maps, masks, titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm, shading = "auto")

    add_stipple(ax, lon, lat, mask, density = 0.05, size = 4, alpha = 0.7)
    ax.set_title(title)

cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.07)
cbar2.set_label("Change in Rain Event Frequency")
fig2.savefig("cesm2_rain_mean_change.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 3 — SIGNAL
# =====================================================
fig3, axs3 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax, data, title in zip(axs3, maps, titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p3 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm)

    ax.set_title("Signal: "+title)

cbar3 = fig3.colorbar(p3, ax = axs3, orientation = "horizontal", pad = 0.07)
cbar3.set_label("Signal (Ensemble Mean Rain Change)")
fig3.savefig("cesm2_rain_signal.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 4 — NOISE
# =====================================================
fig4, axs4 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

noise_maps = [noise1,noise2,noise3]
vmax_noise = max(float(m.max().compute()) for m in noise_maps)

for ax, data, title in zip(axs4,noise_maps,titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p4 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "magma", vmin = 0, vmax = vmax_noise)

    ax.set_title("Noise: "+title)

cbar4 = fig4.colorbar(p4, ax = axs4, orientation = "horizontal", pad = 0.07)
cbar4.set_label("Internal Variability (σ across ensemble)")
fig4.savefig("cesm2_rain_noise.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =====================================================
# FIGURE 5 — SNR MAPS  (ADDED — MISSING IN YOUR RAIN)
# =====================================================
fig5, axs5 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

snr_maps = [snr1,snr2,snr3]

vmin = min(float(m.min().compute()) for m in snr_maps)
vmax = max(float(m.max().compute()) for m in snr_maps)
lim = min(max(abs(vmin), abs(vmax)), 5)

norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

for ax, data, title in zip(axs5, snr_maps, titles):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p5 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm)

    ax.set_title("SNR: "+title)

cbar5 = fig5.colorbar(p5, ax = axs5, orientation = "horizontal", pad = 0.07)
cbar5.set_label("Signal-to-Noise Ratio")

fig5.savefig("cesm2_rain_snr.png", dpi = 300, bbox_inches = "tight")
plt.show()

## ROSI Mask 1970-2100

In [ ]:
# Done - Confirmed
# ======================================================
# DIRECTORIES (YOU PROVIDED THESE)
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

# ======================================================
# VARIABLES & THRESHOLDS
# ======================================================
ice_var  = "aice_d"
snow_var = "vsnon_d"
rain_var = "rain_d"

ice_thresh  = 0.30   # ≥30% ice
snow_thresh = 0.10   # ≥0.1 m snow
rain_thresh = 0.10   # ≥1 mm/day rain

# ======================================================
# MAP SETUP
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

def setup_map(ax):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)
    return ax

# ======================================================
# RoS COMPUTATION (ROBUST TO CESM ISSUES)
# ======================================================
def compute_ros_fraction(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time": 30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time": 30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time": 30}) as ds_r:

        ice  = ds_i[ice_var]
        snow = ds_s[snow_var]
        rain = ds_r[rain_var]

        # ---- Remove duplicate time stamps (CESM bug)
        ice  = ice.isel(time = ~ice.get_index("time").duplicated())
        snow = snow.isel(time = ~snow.get_index("time").duplicated())
        rain = rain.isel(time = ~rain.get_index("time").duplicated())

        # ---- Force positional alignment (NO reindexing)
        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ros = ((ice  >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh))

        return ros.mean(dim = "time")

# ======================================================
# MAIN PIPELINE
# ======================================================
def run_ros_pipeline(ens_tag, VERIFY_PLOTS=True):

    print(f"Running RoS ensemble {ens_tag}")

    # ---- File paths (your naming convention)
    periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

    ice_files = [f"{ICE_DIR}/aice_{p}_{ens_tag}.nc" for p in periods]
    snow_files = [f"{SNOW_DIR}/snow_{p}_{ens_tag}.nc" for p in periods]
    rain_files = [f"{RAIN_DIR}/rain_{p}_{ens_tag}.nc" for p in periods]

    # ---- Compute RoS maps
    ros_maps = [compute_ros_fraction(i, s, r) for i, s, r in zip(ice_files, snow_files, rain_files)]

    lon = ros_maps[0].lon
    lat = ros_maps[0].lat

    # ==================================================
    # FIGURE 1 — RoS FREQUENCY
    # ==================================================
    fig1, axs = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

    titles = [f"RoS Frequency (1970–2000)\nEns {ens_tag}", "RoS Frequency (2000–2025)",
              "RoS Frequency (2025–2065)",  "RoS Frequency (2065–2100)"]

    for ax, data, title in zip(axs.flat, ros_maps, titles):
        setup_map(ax)
        p = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "cividis",  vmin = 0, vmax = 0.05, shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar = fig1.colorbar(p, ax = axs.ravel().tolist(), shrink = 0.78, pad = 0.02)
    cbar.set_label("RoS Frequency (Fraction of Days)")

    # fig1.savefig(f"cesm2_ros_frequency_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig1)

    # ==================================================
    # FIGURE 2 — DIFFERENCES (vs 1970–2000)
    # ==================================================
    diffs = [ros_maps[i] - ros_maps[0] for i in range(1, 4)]
    diff_titles = [
        "Change (2000–2025 − 1970–2000)",
        "Change (2025–2065 − 1970–2000)",
        "Change (2065–2100 − 1970–2000)"
    ]

    lim = max(float(abs(d).max()) for d in diffs)
    norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

    fig2, axs2 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

    for ax, data, title in zip(axs2, diffs, diff_titles):
        setup_map(ax)
        p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm,shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", shrink = 0.85, pad = 0.08)
    cbar2.set_label("Change in RoS Frequency")

    fig2.savefig(f"cesm2_ros_diff_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig2)

# ======================================================
# RUN (BATCH OR INTERACTIVE)
# ======================================================
ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005"]

# 🔹 Set True to SEE plots, False for batch
SHOW_PLOTS = True

for ens in ensemble_members:
    run_ros_pipeline(ens, VERIFY_PLOTS = SHOW_PLOTS)

In [ ]:
# Done - Confirmed
# ======================================================
# DIRECTORIES (YOU PROVIDED THESE)
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

# ======================================================
# VARIABLES & THRESHOLDS
# ======================================================
ice_var  = "aice_d"
snow_var = "vsnon_d"
rain_var = "rain_d"

ice_thresh  = 0.30   # ≥30% ice
snow_thresh = 0.10   # ≥0.1 m snow
rain_thresh = 0.10   # ≥1 mm/day rain

# ======================================================
# MAP SETUP
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

def setup_map(ax):
    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 6)
    ax.gridlines(draw_labels = False, linewidth = 1.0, color = "gray", alpha = 0.8)
    return ax

# ======================================================
# RoS COMPUTATION (ROBUST TO CESM ISSUES)
# ======================================================
def compute_ros_fraction(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time": 30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time": 30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time": 30}) as ds_r:

        ice  = ds_i[ice_var]
        snow = ds_s[snow_var]
        rain = ds_r[rain_var]

        # ---- Remove duplicate time stamps (CESM bug)
        ice  = ice.isel(time = ~ice.get_index("time").duplicated())
        snow = snow.isel(time = ~snow.get_index("time").duplicated())
        rain = rain.isel(time = ~rain.get_index("time").duplicated())

        # ---- Force positional alignment (NO reindexing)
        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ros = ((ice  >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh))

        return ros.mean(dim = "time")

# ======================================================
# MAIN PIPELINE
# ======================================================
def run_ros_pipeline(ens_tag, VERIFY_PLOTS=True):

    print(f"Running RoS ensemble {ens_tag}")

    # ---- File paths (your naming convention)
    periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

    ice_files = [f"{ICE_DIR}/aice_{p}_{ens_tag}.nc" for p in periods]
    snow_files = [f"{SNOW_DIR}/snow_{p}_{ens_tag}.nc" for p in periods]
    rain_files = [f"{RAIN_DIR}/rain_{p}_{ens_tag}.nc" for p in periods]

    # ---- Compute RoS maps
    ros_maps = [compute_ros_fraction(i, s, r) for i, s, r in zip(ice_files, snow_files, rain_files)]

    lon = ros_maps[0].lon
    lat = ros_maps[0].lat

    # ==================================================
    # FIGURE 1 — RoS FREQUENCY
    # ==================================================
    fig1, axs = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

    titles = [f"RoS Frequency (1970–2000)\nEns {ens_tag}", "RoS Frequency (2000–2025)",
              "RoS Frequency (2025–2065)",  "RoS Frequency (2065–2100)"]

    for ax, data, title in zip(axs.flat, ros_maps, titles):
        setup_map(ax)
        p = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "cividis", 
                          vmin = 0, vmax = 0.05, shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar = fig1.colorbar(p, ax = axs.ravel().tolist(), shrink = 0.78, pad = 0.02)
    cbar.set_label("RoS Frequency (Fraction of Days)")

    # fig1.savefig(f"cesm2_ros_frequency_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig1)

    # ==================================================
    # FIGURE 2 — DIFFERENCES (vs 1970–2000)
    # ==================================================
    diffs = [ros_maps[i] - ros_maps[0] for i in range(1, 4)]
    diff_titles = [
        "Change (2000–2025 − 1970–2000)",
        "Change (2025–2065 − 1970–2000)",
        "Change (2065–2100 − 1970–2000)"
    ]

    lim = max(float(abs(d).max()) for d in diffs)
    norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

    fig2, axs2 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

    for ax, data, title in zip(axs2, diffs, diff_titles):
        setup_map(ax)
        p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "coolwarm", norm = norm,shading = "auto")
        ax.set_title(title, fontsize = 13)

    cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", shrink = 0.85, pad = 0.08)
    cbar2.set_label("Change in RoS Frequency")

    fig2.savefig(f"cesm2_ros_diff_{ens_tag}.png", dpi = 300, bbox_inches = "tight")

    if VERIFY_PLOTS:
        plt.show()
    plt.close(fig2)

# ======================================================
# RUN (BATCH OR INTERACTIVE)
# ======================================================
ensemble_members = ["1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

# 🔹 Set True to SEE plots, False for batch
SHOW_PLOTS = True

for ens in ensemble_members:
    run_ros_pipeline(ens, VERIFY_PLOTS = SHOW_PLOTS)

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# DIRECTORIES
# ======================================================
ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

# ======================================================
# THRESHOLDS
# ======================================================
ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10
snr_threshold = 1.0

# ======================================================
# MAP SETTINGS
# ======================================================
proj = ccrs.NorthPolarStereo()
transform = ccrs.PlateCarree()
extent = [-180, 180, 50, 90]

# ======================================================
# LOAD FUNCTION — RETURNS ANNUAL FRACTION
# ======================================================
def annual_ros_fraction(ice_file, snow_file, rain_file):

    with xr.open_dataset(ice_file, chunks = {"time":30}) as ds_i, \
         xr.open_dataset(snow_file, chunks = {"time":30}) as ds_s, \
         xr.open_dataset(rain_file, chunks = {"time":30}) as ds_r:

        ice  = ds_i["aice_d"]
        snow = ds_s["vsnon_d"]
        rain = ds_r["rain_d"]

        # remove duplicate CESM times
        ice  = ice.isel(time = ~ice.get_index("time").duplicated())
        snow = snow.isel(time = ~snow.get_index("time").duplicated())
        rain = rain.isel(time = ~rain.get_index("time").duplicated())

        ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

        ros = (ice >= ice_thresh) & (snow >= snow_thresh) & (rain >= rain_thresh)

        # ---- CRITICAL: KEEP YEAR DIMENSION ----
        yearly = ros.groupby("time.year").mean("time")

        return yearly

# ======================================================
# LOAD ALL ENSEMBLES
# ======================================================
ros_all = {p:[] for p in periods}

for ens in ensemble_members:
    print("Processing ensemble:",ens)
    for p in periods:
        ros_all[p].append(annual_ros_fraction(
            f"{ICE_DIR}/aice_{p}_{ens}.nc", f"{SNOW_DIR}/snow_{p}_{ens}.nc", f"{RAIN_DIR}/rain_{p}_{ens}.nc"))

for p in periods:
    ros_all[p] = xr.concat(ros_all[p], dim = "ens").assign_coords(ens = ensemble_members)

lon = ros_all[periods[0]].lon
lat = ros_all[periods[0]].lat

# ======================================================
# COLLAPSE HELPER
# ======================================================
def collapse_state(frac):
    return frac.mean(dim = [d for d in frac.dims if d in ["ens", "year"]])

# ======================================================
# DISCRETE COLOR LEVELS (MATCH OBS)
# ======================================================
levels = [0, 0.001, 0.003, 0.007, 0.012, 0.018, 0.025, 0.03]

cmap = plt.get_cmap("RdYlBu_r")
norm = BoundaryNorm(levels, cmap.N)

# ======================================================
# FIGURE 1 — MEAN STATE
# ======================================================
means = [collapse_state(ros_all[p]) for p in periods]

fig1, axs1 = plt.subplots(2, 2, figsize = (12, 10), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax, data, title in zip(axs1.flat, means, periods):

    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p1 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = cmap, norm = norm, shading = "auto")

    ax.set_title(title, fontsize = 13)

# ======================================================
# DISCRETE COLORBAR
# ======================================================
cbar1 = fig1.colorbar(p1, ax = axs1.ravel().tolist(), orientation = "vertical", pad = 0.02, 
                      shrink = 0.85, extend = "max", boundaries = levels)

cbar1.set_label("ROSI Occurrence (fraction of days)", size = 12)
cbar1.set_ticks(levels)
cbar1.ax.tick_params(labelsize = 11)

fig1.suptitle("Pan-Arctic Mean Rain-on-Snow-on-Ice Occurrence", fontsize = 16, y = 1.02)

fig1.savefig("cesm2_rosi_mean_state.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ======================================================
# SIGNAL / NOISE
# ======================================================
mean1_ens = ros_all["1970-2000"].mean("year")
mean2_ens = ros_all["2000-2025"].mean("year")
mean3_ens = ros_all["2025-2065"].mean("year")
mean4_ens = ros_all["2065-2100"].mean("year")

signal1 = mean2_ens.mean("ens") - mean1_ens.mean("ens")
signal2 = mean4_ens.mean("ens") - mean3_ens.mean("ens")
signal3 = mean4_ens.mean("ens") - mean1_ens.mean("ens")

noise1 = (mean2_ens - mean1_ens).std("ens")
noise2 = (mean4_ens - mean3_ens).std("ens")
noise3 = (mean4_ens - mean1_ens).std("ens")

eps = 1e-6
snr1 = signal1 / noise1.where(noise1 > eps)
snr2 = signal2 / noise2.where(noise2 > eps)
snr3 = signal3 / noise3.where(noise3 > eps)

mask1 = np.abs(snr1) > snr_threshold
mask2 = np.abs(snr2) > snr_threshold
mask3 = np.abs(snr3) > snr_threshold

print("SNR1 range:", float(snr1.min().compute()), float(snr1.max().compute()))
print("SNR2 range:", float(snr2.min().compute()), float(snr2.max().compute()))
print("SNR3 range:", float(snr3.min().compute()), float(snr3.max().compute()))

print("\nNoise1 min/max:", float(noise1.min().compute()), float(noise1.max().compute()))
print("Noise2 min/max:", float(noise2.min().compute()), float(noise2.max().compute()))
print("Noise3 min/max:", float(noise3.min().compute()), float(noise3.max().compute()))

print("\nSignal1 min/max:", float(signal1.min().compute()), float(signal1.max().compute()))
print("Signal2 min/max:", float(signal2.min().compute()), float(signal2.max().compute()))
print("Signal3 min/max:", float(signal3.min().compute()), float(signal3.max().compute()))

# ======================================================
# HISTOGRAMS
# ======================================================
sig = signal3

vals_all = sig.data.compute().ravel()
vals_all = vals_all[np.isfinite(vals_all)]

baseline = mean1_ens.mean("ens")
ros_mask = baseline > 0.005   # rare-event filter

vals_ros = sig.where(ros_mask).data.compute().ravel()
vals_ros = vals_ros[np.isfinite(vals_ros)]

fig, axs = plt.subplots(2, 1, figsize = (8, 8), constrained_layout = True)

axs[0].hist(vals_all, bins = 80)
axs[0].axvline(0, linestyle = "--")
axs[0].set_title("Change distribution — ALL grid cells")

axs[1].hist(vals_ros, bins = 80)
axs[1].axvline(0, linestyle = "--")
axs[1].set_title("Change distribution — RoS cells only")

plt.savefig("cesm2_rosi_histograms.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ======================================================
# STIPPLE FUNCTION 
# ======================================================
def add_stipple(ax, lon_da, lat_da, mask_da, density = 0.12, jitter = 0.3, size = 6, alpha = 0.9, color = "tomato", seed = 42):

    np.random.seed(seed)
    ys, xs = np.where(mask_da.values)
    if len(xs) == 0:
        return

    if getattr(lon_da, "ndim", None) == 2:
        sig_lon = lon_da.values[ys, xs]
        sig_lat = lat_da.values[ys, xs]
    else:
        sig_lon = lon_da.values[xs]
        sig_lat = lat_da.values[ys]

    keep = np.random.rand(len(sig_lon)) < density
    if keep.sum() == 0:
        return

    sig_lon = sig_lon[keep] + np.random.uniform(-jitter, jitter, keep.sum())
    sig_lat = sig_lat[keep] + np.random.uniform(-jitter, jitter, keep.sum())

    ax.scatter(sig_lon, sig_lat, s = size, marker = ".", color = color, alpha = alpha, transform = ccrs.PlateCarree(), zorder = 30)

# ======================================================
# FIGURE 2 — CHANGE MAPS + STIPPLING
# ======================================================
maps = [signal1, signal2, signal3]
masks = [mask1, mask2, mask3]
titles = ["2000–2025 - 1970–2000", "2065–2100 - 2025–2065", "2065–2100 - 1970–2000"]

vmin = min(float(m.min().compute()) for m in maps)
vmax = max(float(m.max().compute()) for m in maps)
lim = max(abs(vmin), abs(vmax))
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)

fig2, axs2 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax, data, mask, title in zip(axs2, maps, masks, titles):

    ax.set_extent(extent,crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9, zorder = 25)

    p2 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "RdBu_r", norm = norm, shading = "auto")

    add_stipple(ax, lon, lat, mask, density = 0.05, size = 4, alpha = 0.7)
    ax.set_title(title)

cbar2 = fig2.colorbar(p2, ax = axs2, orientation = "horizontal", pad = 0.08, shrink = 0.85)
cbar2.set_label("Δ ROSI Occurrence (fraction of days)", fontsize = 12)
cbar2.ax.tick_params(labelsize = 11)

fig2.suptitle("Ensemble-Mean Change in Rain-on-Snow-on-Ice Occurrence", fontsize = 16, y = 1.03)

fig2.savefig("cesm2_rosi_mean_change.png", dpi = 300, bbox_inches = "tight")
plt.show()

# =========================
# FIGURE 3 — NOISE MAPS
# =========================
noises = [noise1, noise2, noise3]
titles_noise = ["2000–2025 - 1970–2000", "2065–2100 - 2025–2065", "2065–2100 - 1970–2000"]

# dask-safe global max for consistent color scale
vmax_noise = max(float(n.max().compute()) for n in noises)

fig3, axs3 = plt.subplots(1, 3, figsize = (15, 5), subplot_kw = {'projection': proj}, constrained_layout = True)

for ax, data, title in zip(axs3, noises, titles_noise):

    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    # plot noise field
    pcm = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "magma", vmin = 0, vmax = vmax_noise, shading = "auto")

    ax.set_title(title)

# shared colorbar
cbar3 = fig3.colorbar(pcm, ax = axs3, orientation = "horizontal", pad = 0.07)
cbar3.set_label("Ensemble Standard Deviation (fraction of days)", fontsize = 12)
cbar3.ax.tick_params(labelsize = 11)

fig3.suptitle("Ensemble Standard Deviation of Rain-on-Snow-on-Ice Change", fontsize = 16, y = 1.05)

fig3.savefig("cesm2_rosi_noise.png", dpi = 300, bbox_inches = "tight")
plt.show()

# ======================================================
# FIGURE 4 — SNR MAPS
# ======================================================
snr_maps = [snr1, snr2, snr3]

vmin = min(float(m.min().compute()) for m in snr_maps)
vmax = max(float(m.max().compute()) for m in snr_maps)
lim = min(max(abs(vmin), abs(vmax)), 5)
norm = TwoSlopeNorm(vcenter = 0, vmin = -lim, vmax = lim)
titles_snr = ["2000–2025 - 1970–2000", "2065–2100 - 2025–2065", "2065–2100 - 1970–2000"]

fig4, axs4 = plt.subplots(1, 3, figsize = (18, 6), subplot_kw = dict(projection = proj), constrained_layout = True)

for ax, data, title in zip(axs4, snr_maps, titles_snr):

    ax.set_extent(extent, crs = transform)
    ax.add_feature(cfeature.LAND, facecolor = "0.75", edgecolor = "k", linewidth = 0.3, zorder = 5)
    ax.coastlines(linewidth = 0.9)

    p4 = ax.pcolormesh(lon, lat, data, transform = transform, cmap = "RdBu_r", norm = norm, shading = "auto")

    ax.set_title(title, fontsize = 13)

cbar4 = fig4.colorbar(p4, ax = axs4, orientation = "horizontal", pad = 0.07)
cbar4.set_label("Signal-to-Noise Ratio (SNR)", fontsize = 12)
cbar4.ax.tick_params(labelsize = 11)

fig4.suptitle("Signal-to-Noise Ratio of Rain-on-Snow-on-Ice Change", fontsize = 16, y = 1.05)

fig4.savefig("cesm2_rosi_snr.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
warnings.filterwarnings("ignore", category = RuntimeWarning)

# ======================================================
# TIME SERIES (ALL 10)
# ======================================================

ICE_DIR  = "/glade/work/xliu1/cesm2_seaice_regrid"
SNOW_DIR = "/glade/work/xliu1/cesm2_sosi_regrid"
RAIN_DIR = "/glade/work/xliu1/cesm2_rosi_regrid"

ensemble_members = ["1011.001", "1031.002", "1051.003", "1071.004", "1091.005",
                    "1111.006", "1131.007", "1151.008", "1171.009", "1191.010"]

periods = ["1970-2000", "2000-2025", "2025-2065", "2065-2100"]

ice_thresh  = 0.30
snow_thresh = 0.10
rain_thresh = 0.10

# ======================================================
# FUNCTION: Compute annual RoS frequency for one file set
# ======================================================
def compute_ros_timeseries(ice_file, snow_file, rain_file):

    ds_i = xr.open_dataset(ice_file)
    ds_s = xr.open_dataset(snow_file)
    ds_r = xr.open_dataset(rain_file)

    ice  = ds_i["aice_d"]
    snow = ds_s["vsnon_d"]
    rain = ds_r["rain_d"]

    ice  = ice.isel(time=~ice.get_index("time").duplicated())
    snow = snow.isel(time=~snow.get_index("time").duplicated())
    rain = rain.isel(time=~rain.get_index("time").duplicated())

    ice, snow, rain = xr.align(ice, snow, rain, join = "inner")

    weights = np.cos(np.deg2rad(ice.lat.values))[:, None]
    weights = weights / np.nanmean(weights)

    ros_daily = []

    for t in range(len(ice.time)):

        ice_mask = ice.isel(time=t).values >= ice_thresh

        ros_t = (
            ice_mask
            & (snow.isel(time=t).values >= snow_thresh)
            & (rain.isel(time=t).values >= rain_thresh)
        )

        ice_area = np.nansum(ice_mask * weights)

        if ice_area > 0:
            val = np.nansum(ros_t * weights) / ice_area
        else:
            val = 0.0

        ros_daily.append(val)

    ds_i.close(); ds_s.close(); ds_r.close()

    return xr.DataArray(ros_daily, coords = {"time": ice.time}, dims = "time")

# ======================================================
# BUILD ENSEMBLE TIMESERIES
# ======================================================
ros_ensemble = []

for ens in ensemble_members:

    print(f"Processing ensemble {ens}")
    member_series = []

    for period in periods:

        ice_file  = os.path.join(ICE_DIR,  f"aice_{period}_{ens}.nc")
        snow_file = os.path.join(SNOW_DIR, f"snow_{period}_{ens}.nc")
        rain_file = os.path.join(RAIN_DIR, f"rain_{period}_{ens}.nc")

        ts = compute_ros_timeseries(ice_file, snow_file, rain_file)
        member_series.append(ts)

    # concatenate DAILY first
    member_daily = xr.concat(member_series, dim = "time")
    member_daily = member_daily.sortby("time")
    member_daily = member_daily.sel(time = ~member_daily.get_index("time").duplicated())

    # ensure continuous daily timeline
    full_time = xr.cftime_range(start = member_daily.time.values.min(), end = member_daily.time.values.max(), freq = "D")
    member_daily = member_daily.reindex(time = full_time)

    # THEN compute annual
    annual_mean = member_daily.resample(time = "YE").mean(skipna = True)
    annual_days = member_daily.resample(time = "YE").count()

    ens_ts = annual_mean.where(annual_days >= 300)

    print(ens, "years:", ens_ts.sizes["time"], "nan years:", int(np.isnan(ens_ts).sum()))

    ros_ensemble.append(ens_ts)

# Stack ensembles
ros_ts_all = xr.concat(ros_ensemble, dim = "ens", join = "inner")

if not isinstance(ros_ts_all.indexes["time"], pd.DatetimeIndex):
    ros_ts_all = ros_ts_all.assign_coords(time = ros_ts_all.indexes["time"].to_datetimeindex())

ros_ts_all = ros_ts_all.sel(time = slice("1970", "2100"))

valid = ros_ts_all.count("ens") >= 5
mean_ts = ros_ts_all.mean("ens")
std_ts = ros_ts_all.std("ens").where(valid)

print("\nMin members per year:", ros_ts_all.count("ens").min().values)

member_count = ros_ts_all.count("ens")
print("\nYears with <5 members:", pd.to_datetime(mean_ts.time.values)[member_count.values < 5].year)

print("\nMembers:", ros_ts_all.sizes["ens"])
print("Years:", ros_ts_all.sizes["time"])
print("Any NaNs:", np.isnan(ros_ts_all).sum().values)
print("Min std:", float(std_ts.min()))
print("Max std:", float(std_ts.max()))

# ======================================================
# PLOT
# ======================================================
plt.figure(figsize = (11, 5))

time = mean_ts["time"].values

plt.plot(time, mean_ts.values, linewidth = 2, label = "Ensemble Mean")

plt.fill_between(time, (mean_ts - std_ts).values, (mean_ts + std_ts).values, alpha = 0.3, label = "Ensemble spread (±1σ)")

plt.title("Pan-Arctic Rain-on-Snow-on-Ice Frequency (CESM2-LE)")
plt.ylabel("Fraction of Arctic grid cells with ROSI events")
plt.grid(alpha = 0.3)
plt.legend()
plt.tight_layout()
plt.savefig("cesm2_rosi_timeseries.png", dpi = 300, bbox_inches = "tight")
plt.show()

# year-to-year change
dy = mean_ts.diff("time")

# Top positive spikes
largest_up = dy.sortby(dy, ascending = False).isel(time = slice(0, 5))

# Top negative spikes
largest_down = dy.sortby(dy).isel(time = slice(0, 5))

# Largest year to year spikes
print("Largest increases:")
for t, v in zip(largest_up.time.values, largest_up.values):
    print(pd.to_datetime(t).year, round(float(v)*100, 2), "%")

print("\nLargest decreases:")
for t, v in zip(largest_down.time.values, largest_down.values):
    print(pd.to_datetime(t).year, round(float(v)*100, 2), "%")

# Long-term change climate signal
baseline = mean_ts.sel(time = slice("1970", "2000")).mean()
future   = mean_ts.sel(time = slice("2065", "2100")).mean()

change = float((future - baseline) * 100)
print("\nLong-term RoS increase:", round(change, 2), "% of Arctic area")

# Trend per decade
years = (mean_ts.time.dt.year - mean_ts.time.dt.year[0]).values
mask = np.isfinite(mean_ts.values)
coef = np.polyfit(years[mask], mean_ts.values[mask], 1)

trend_per_decade = coef[0] * 10 * 100
print("\nTrend:", round(trend_per_decade, 2), "% Arctic area per decade")

# Statistically extreme years
z = (mean_ts - mean_ts.mean()) / mean_ts.std()
extreme = mean_ts.time.where(abs(z) > 2, drop = True)

print("\nExtreme RoS years:")
print(pd.to_datetime(extreme.values).year)

In [ ]:
# ======================================================
# PLOT EACH ENSEMBLE MEMBER
# ======================================================
time = ros_ts_all.time.values
ens_mean = ros_ts_all.mean("ens")

for i in range(ros_ts_all.sizes["ens"]):

    plt.figure(figsize = (10, 4))

    member = ros_ts_all.isel(ens = i)

    plt.plot(time, ens_mean.values, linewidth = 2, linestyle = "--", label = "Forced signal")
    plt.plot(time, member.values, linewidth = 1.5, label = "Member")

    plt.fill_between(time, ens_mean.values, member.values, alpha = 0.2)

    plt.title(f"CESM2-LE RoS + Internal Variability — {ensemble_members[i]}")
    plt.ylabel("RoS fraction")
    plt.xlabel("Year")
    plt.legend()
    plt.grid(alpha = 0.3)

    plt.tight_layout()
    plt.show()